In [1]:
import pandas as pd
import numpy as np

geno_path = r"C:\Users\user\Downloads\GSE148375_clean\checkpoint0_snp_clean_v2.txt"

chunksize = 20000
sample_nc_counts = None
sample_cols = None
total_probes = 0

reader = pd.read_csv(geno_path, sep="\t", chunksize=chunksize)

for chunk in reader:
    if sample_cols is None:
        sample_cols = chunk.columns[1:]
        sample_nc_counts = pd.Series(0, index=sample_cols)

    total_probes += len(chunk)
    nc_mask = (chunk[sample_cols] == "NC")
    sample_nc_counts += nc_mask.sum(axis=0)

sample_missing_rate = sample_nc_counts / total_probes

print("Total probes scanned:", total_probes)
print("Missing rate summary (per sample):")
print(sample_missing_rate.describe())

Total probes scanned: 242764
Missing rate summary (per sample):
count    3396.000000
mean        0.006329
std         0.006313
min         0.000906
25%         0.003464
50%         0.004436
75%         0.007399
max         0.245712
dtype: float64


In [2]:
# look at the upper tail in detail
print(sample_missing_rate.sort_values(ascending=False).head(20))

# how many samples exceed various candidate thresholds
for thresh in [0.01, 0.02, 0.03, 0.05, 0.10]:
    n_exceed = (sample_missing_rate > thresh).sum()
    print(f"Samples with missingness > {thresh*100:.0f}%: {n_exceed}")

215753     0.245712
2100285    0.047532
2104054    0.045167
2104088    0.045106
2104206    0.044998
2100271    0.043800
2100834    0.043664
2104207    0.043606
2100277    0.043272
2100284    0.043260
2104222    0.042910
2104113    0.042737
2100873    0.042712
2104227    0.042609
2100900    0.042589
2100874    0.042510
2104214    0.042461
2100842    0.042440
2100780    0.042420
2104228    0.042370
dtype: float64
Samples with missingness > 1%: 641
Samples with missingness > 2%: 48
Samples with missingness > 3%: 34
Samples with missingness > 5%: 1
Samples with missingness > 10%: 1


In [3]:
# finer breakdown between 1% and 5%
for thresh in [0.015, 0.02, 0.025, 0.03, 0.035, 0.04, 0.045]:
    n_exceed = (sample_missing_rate > thresh).sum()
    print(f"> {thresh*100:.1f}%: {n_exceed}")

# check if the high-missingness cluster shares an ID prefix (possible batch effect)
high_missing = sample_missing_rate[sample_missing_rate > 0.02].sort_values(ascending=False)
print(high_missing)

> 1.5%: 75
> 2.0%: 48
> 2.5%: 38
> 3.0%: 34
> 3.5%: 29
> 4.0%: 21
> 4.5%: 4
215753     0.245712
2100285    0.047532
2104054    0.045167
2104088    0.045106
2104206    0.044998
2100271    0.043800
2100834    0.043664
2104207    0.043606
2100277    0.043272
2100284    0.043260
2104222    0.042910
2104113    0.042737
2100873    0.042712
2104227    0.042609
2100900    0.042589
2100874    0.042510
2104214    0.042461
2100842    0.042440
2100780    0.042420
2104228    0.042370
2100797    0.042309
2101839    0.039759
2100453    0.038424
2101867    0.037988
2101381    0.036146
2101382    0.036134
2101593    0.035668
2101823    0.035273
2101594    0.035224
2100988    0.034968
2105081    0.034099
2104063    0.033197
2104064    0.032159
2105082    0.030824
2102709    0.029411
2101178    0.028847
2104502    0.028526
2104501    0.026742
2102690    0.024958
2105079    0.024954
2100335    0.024785
2100268    0.024707
2100363    0.023879
2100336    0.023257
2100195    0.021791
2103976    0.021626
2100

In [4]:
import pandas as pd
import os

geno_path = r"C:\Users\user\Downloads\GSE148375_clean\checkpoint0_snp_clean_v2.txt"
out_dir = r"C:\Users\user\Downloads\GSE148375_clean"

threshold = 0.02
samples_to_drop = sample_missing_rate[sample_missing_rate > threshold].index.tolist()

print("Number of samples to drop:", len(samples_to_drop))
print(samples_to_drop)

# save the dropped list for the QC log
pd.Series(samples_to_drop, name="dropped_sample_id").to_csv(
    os.path.join(out_dir, "qc_log_dropped_samples_missingness.csv"), index=False
)
print("Saved dropped-sample log.")

Number of samples to drop: 48
['2100162', '2100195', '2100268', '2100271', '2100277', '2100284', '2100285', '2100335', '2100336', '2100363', '2100453', '2100465', '2100780', '2100797', '2100834', '2100842', '2100873', '2100874', '2100900', '2100988', '2101178', '2101381', '2101382', '2101593', '2101594', '2101823', '2101839', '2101867', '2102690', '2102709', '2103976', '2104054', '2104063', '2104064', '2104088', '2104113', '2104206', '2104207', '2104214', '2104222', '2104227', '2104228', '2104501', '2104502', '2105079', '2105081', '2105082', '215753']
Saved dropped-sample log.


In [5]:
import pandas as pd
import os

geno_path = r"C:\Users\user\Downloads\GSE148375_clean\checkpoint0_snp_clean_v2.txt"
meta_path = r"C:\Users\user\Downloads\GSE148375_clean\checkpoint1_metadata_binary.csv"
out_dir = r"C:\Users\user\Downloads\GSE148375_clean"

samples_to_drop_set = set(samples_to_drop)

# --- Rebuild genotype file without these sample columns ---
chunksize = 20000
out_path = os.path.join(out_dir, "checkpoint2_snp_sample_filtered.txt")
first_chunk = True

reader = pd.read_csv(geno_path, sep="\t", chunksize=chunksize)

for chunk in reader:
    cols_to_drop = [c for c in chunk.columns if c in samples_to_drop_set]
    chunk = chunk.drop(columns=cols_to_drop)
    chunk.to_csv(out_path, sep="\t", mode="w" if first_chunk else "a",
                 header=first_chunk, index=False)
    first_chunk = False

print("Saved sample-filtered genotype checkpoint:", out_path)

# --- Sync metadata ---
meta_df_clean = pd.read_csv(meta_path)
meta_df_clean["sample_id"] = meta_df_clean["sample_id"].astype(str)

meta_df_filtered = meta_df_clean[~meta_df_clean["sample_id"].isin(samples_to_drop_set)].copy()
print("Metadata shape after dropping:", meta_df_filtered.shape)
# expect 3396 - 48 = 3348

meta_df_filtered.to_csv(os.path.join(out_dir, "checkpoint2_metadata_sample_filtered.csv"), index=False)
print("Saved sample-filtered metadata checkpoint.")

Saved sample-filtered genotype checkpoint: C:\Users\user\Downloads\GSE148375_clean\checkpoint2_snp_sample_filtered.txt
Metadata shape after dropping: (3348, 9)
Saved sample-filtered metadata checkpoint.


In [6]:
out_path = r"C:\Users\user\Downloads\GSE148375_clean\checkpoint2_snp_sample_filtered.txt"

with open(out_path, encoding="utf-8") as f:
    header = f.readline()
    n_cols = len(header.strip().split("\t"))
    n_rows = sum(1 for _ in f)

print("Genotype checkpoint2 shape (rows, cols):", n_rows, n_cols)
# expect 242764 rows, 3349 cols (3348 samples + 1 ID col)

Genotype checkpoint2 shape (rows, cols): 242764 3349


In [7]:
import pandas as pd
import numpy as np

geno_path = r"C:\Users\user\Downloads\GSE148375_clean\checkpoint2_snp_sample_filtered.txt"

chunksize = 20000
probe_missing_rates = []
probe_ids = []

reader = pd.read_csv(geno_path, sep="\t", chunksize=chunksize)

for chunk in reader:
    id_col = chunk.columns[0]
    geno_cols = chunk.columns[1:]
    n_samples = len(geno_cols)

    nc_count = (chunk[geno_cols] == "NC").sum(axis=1)
    missing_rate = nc_count / n_samples

    probe_ids.extend(chunk[id_col].tolist())
    probe_missing_rates.extend(missing_rate.tolist())

probe_missing_series = pd.Series(probe_missing_rates, index=probe_ids)

print("Total probes:", len(probe_missing_series))
print(probe_missing_series.describe())

Total probes: 242764
count    242764.000000
mean          0.005846
std           0.050772
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max           1.000000
dtype: float64


In [8]:
# finer percentiles since most mass is at 0
for p in [0.90, 0.95, 0.97, 0.99, 0.995, 0.999]:
    print(f"{p*100:.1f}th percentile: {probe_missing_series.quantile(p):.4f}")

print()
for thresh in [0.01, 0.02, 0.05, 0.10, 0.20, 0.50]:
    n_exceed = (probe_missing_series > thresh).sum()
    print(f"Probes with missingness > {thresh*100:.0f}%: {n_exceed} ({100*n_exceed/len(probe_missing_series):.2f}%)")

90.0th percentile: 0.0009
95.0th percentile: 0.0036
97.0th percentile: 0.0087
99.0th percentile: 0.1828
99.5th percentile: 0.3877
99.9th percentile: 0.7875

Probes with missingness > 1%: 6550 (2.70%)
Probes with missingness > 2%: 4718 (1.94%)
Probes with missingness > 5%: 3837 (1.58%)
Probes with missingness > 10%: 3592 (1.48%)
Probes with missingness > 20%: 2188 (0.90%)
Probes with missingness > 50%: 889 (0.37%)


In [9]:
import pandas as pd
import os

geno_path = r"C:\Users\user\Downloads\GSE148375_clean\checkpoint2_snp_sample_filtered.txt"
out_dir = r"C:\Users\user\Downloads\GSE148375_clean"

threshold = 0.05
probes_to_drop = set(probe_missing_series[probe_missing_series > threshold].index)

print("Number of probes to drop:", len(probes_to_drop))

# save QC log
pd.Series(list(probes_to_drop), name="dropped_probe_id").to_csv(
    os.path.join(out_dir, "qc_log_dropped_probes_missingness.csv"), index=False
)
print("Saved dropped-probe log.")

Number of probes to drop: 3837
Saved dropped-probe log.


In [10]:
chunksize = 20000
out_path = os.path.join(out_dir, "checkpoint3_snp_probe_filtered.txt")
first_chunk = True

reader = pd.read_csv(geno_path, sep="\t", chunksize=chunksize)

for chunk in reader:
    id_col = chunk.columns[0]
    chunk = chunk[~chunk[id_col].isin(probes_to_drop)]
    chunk.to_csv(out_path, sep="\t", mode="w" if first_chunk else "a",
                 header=first_chunk, index=False)
    first_chunk = False

print("Saved probe-filtered genotype checkpoint:", out_path)

Saved probe-filtered genotype checkpoint: C:\Users\user\Downloads\GSE148375_clean\checkpoint3_snp_probe_filtered.txt


In [11]:
out_path = r"C:\Users\user\Downloads\GSE148375_clean\checkpoint3_snp_probe_filtered.txt"

with open(out_path, encoding="utf-8") as f:
    header = f.readline()
    n_cols = len(header.strip().split("\t"))
    n_rows = sum(1 for _ in f)

print("Genotype checkpoint3 shape (rows, cols):", n_rows, n_cols)
# expect 238927 rows, 3349 cols


Genotype checkpoint3 shape (rows, cols): 238927 3349


In [13]:
import pandas as pd
import numpy as np

geno_path = r"C:\Users\user\Downloads\GSE148375_clean\checkpoint3_snp_probe_filtered.txt"
out_dir = r"C:\Users\user\Downloads\GSE148375_clean"

chunksize = 20000
alleles = ['A', 'C', 'G', 'T']

results = []

reader = pd.read_csv(geno_path, sep="\t", chunksize=chunksize)

for chunk in reader:
    id_col = chunk.columns[0]
    geno_cols = chunk.columns[1:]
    ids = chunk[id_col].values

    arr = chunk[geno_cols].values.astype('<U2')
    arr = np.ascontiguousarray(arr)
    chars = arr.view('U1').reshape(arr.shape[0], arr.shape[1], 2)
    first, second = chars[:, :, 0], chars[:, :, 1]

    is_nc = (arr == 'NC')
    is_hom = (first == second) & ~is_nc
    is_het = (first != second) & ~is_nc

    n_het = is_het.sum(axis=1)
    n_missing = is_nc.sum(axis=1)

    hom_counts = np.zeros((arr.shape[0], 4), dtype=int)
    for i, letter in enumerate(alleles):
        hom_counts[:, i] = ((first == letter) & is_hom).sum(axis=1)

    sorted_hom = np.sort(hom_counts, axis=1)[:, ::-1]
    n_hom1 = sorted_hom[:, 0]
    n_hom2 = sorted_hom[:, 1]

    for j in range(len(ids)):
        results.append({
            "probe_id": ids[j],
            "n_hom1": n_hom1[j],
            "n_het": n_het[j],
            "n_hom2": n_hom2[j],
            "n_missing": n_missing[j]
        })

counts_df = pd.DataFrame(results)
print(counts_df.shape)
print(counts_df.head())

counts_df.to_csv(os.path.join(out_dir, "checkpoint4_genotype_counts.csv"), index=False)
print("Saved genotype counts checkpoint.")

(238927, 5)
                      probe_id  n_hom1  n_het  n_hom2  n_missing
0  exm2268640-0_B_F_1984844585    2855    470      16          7
1       exm41-0_B_F_1921435147    3348      0       0          0
2  exm1916089-0_B_R_1927689775    3343      0       0          5
3       exm44-0_B_R_1921538602    3141    203       4          0
4       exm46-0_T_F_1921333919    3298     49       0          1
Saved genotype counts checkpoint.


In [14]:
import numpy as np
import pandas as pd
from scipy.stats import hypergeom

def hwe_exact_pvalue(n_hom1, n_het, n_hom2):
    """
    Wigginton et al. (2005) exact HWE test.
    n_hom1: homozygous count for allele 1
    n_het: heterozygous count
    n_hom2: homozygous count for allele 2
    Returns two-sided exact p-value.
    """
    n_hom_rare = min(n_hom1, n_hom2)
    n_hom_common = max(n_hom1, n_hom2)
    n_rare_alleles = 2 * n_hom_rare + n_het  # total copies of rare allele
    n_total_alleles = 2 * (n_hom1 + n_het + n_hom2)

    if n_rare_alleles == 0:
        return 1.0  # monomorphic, no point testing

    # possible heterozygote counts range from 0 (or parity-matched) up to n_rare_alleles
    het_range = np.arange(n_rare_alleles % 2, n_rare_alleles + 1, 2)

    # probability mass function for each possible het count under HWE
    # using the exact combinatorial formula (Wigginton et al.)
    probs = []
    for h in het_range:
        hom_r = (n_rare_alleles - h) // 2
        # log multinomial-style probability, normalized later
        log_p = (np.log(2) * h
                 if False else 0)  # placeholder, will normalize via relative likelihood
        probs.append(h)

    # Simpler robust approach: use scipy's exact HWE via direct enumeration of likelihoods
    obs_het = n_het
    N = n_hom1 + n_het + n_hom2
    rare = n_rare_alleles

    log_probs = []
    hets = np.arange(rare % 2, rare + 1, 2)
    for h in hets:
        homr = (rare - h) // 2
        homc = N - h - homr
        if homc < 0:
            continue
        # log likelihood proportional to multinomial coefficient under HWE
        from scipy.special import gammaln
        log_lik = (gammaln(N + 1) - gammaln(homr + 1) - gammaln(h + 1) - gammaln(homc + 1)
                   + h * np.log(2))
        log_probs.append((h, log_lik))

    hs, log_liks = zip(*log_probs)
    log_liks = np.array(log_liks)
    liks = np.exp(log_liks - log_liks.max())
    liks /= liks.sum()

    obs_idx = list(hs).index(obs_het) if obs_het in hs else None
    if obs_idx is None:
        return np.nan

    obs_lik = liks[obs_idx]
    p_value = liks[liks <= obs_lik + 1e-10].sum()
    return min(p_value, 1.0)

# quick test on a few rows
counts_df = pd.read_csv(r"C:\Users\user\Downloads\GSE148375_clean\checkpoint4_genotype_counts.csv")

test_sample = counts_df.head(10)
for _, row in test_sample.iterrows():
    p = hwe_exact_pvalue(row["n_hom1"], row["n_het"], row["n_hom2"])
    print(row["probe_id"], row["n_hom1"], row["n_het"], row["n_hom2"], "p =", p)

exm2268640-0_B_F_1984844585 2855 470 16 p = 0.5353467059663304
exm41-0_B_F_1921435147 3348 0 0 p = 1.0
exm1916089-0_B_R_1927689775 3343 0 0 p = 1.0
exm44-0_B_R_1921538602 3141 203 4 p = 0.5724424296919856
exm46-0_T_F_1921333919 3298 49 0 p = 0.9999999999999999
exm47-0_T_F_1921489985 3342 6 0 p = 1.0
exm51-0_B_R_1921417192 3294 22 0 p = 1.0
exm53-0_B_R_1921414362 3348 0 0 p = 1.0
exm55-0_T_R_1921549846 3183 165 0 p = 0.268095946598635
exm56-0_T_R_1921349466 3321 27 0 p = 1.0


In [15]:
! pip install tqdm

In [16]:
import numpy as np
from scipy.special import gammaln
import pandas as pd
from tqdm import tqdm

def hwe_exact_pvalue(n_hom1, n_het, n_hom2):
    n_hom_rare = min(n_hom1, n_hom2)
    n_rare_alleles = 2 * n_hom_rare + n_het
    N = n_hom1 + n_het + n_hom2

    if n_rare_alleles == 0:
        return 1.0

    hets = np.arange(n_rare_alleles % 2, n_rare_alleles + 1, 2)
    homr = (n_rare_alleles - hets) // 2
    homc = N - hets - homr
    valid = homc >= 0
    hets, homr, homc = hets[valid], homr[valid], homc[valid]

    log_liks = (gammaln(N + 1) - gammaln(homr + 1) - gammaln(hets + 1) - gammaln(homc + 1)
                + hets * np.log(2))
    liks = np.exp(log_liks - log_liks.max())
    liks /= liks.sum()

    obs_idx = np.where(hets == n_het)[0]
    if len(obs_idx) == 0:
        return np.nan
    obs_lik = liks[obs_idx[0]]
    return min(liks[liks <= obs_lik + 1e-10].sum(), 1.0)

counts_df = pd.read_csv(r"C:\Users\user\Downloads\GSE148375_clean\checkpoint4_genotype_counts.csv")

pvals = np.empty(len(counts_df))
for i, row in enumerate(tqdm(counts_df.itertuples(index=False), total=len(counts_df))):
    pvals[i] = hwe_exact_pvalue(row.n_hom1, row.n_het, row.n_hom2)

counts_df["hwe_pvalue"] = pvals
counts_df.to_csv(r"C:\Users\user\Downloads\GSE148375_clean\checkpoint5_hwe_results.csv", index=False)
print("Done. Saved HWE results.")
print(counts_df["hwe_pvalue"].describe())

100%|██████████| 238927/238927 [00:08<00:00, 28017.22it/s]


Done. Saved HWE results.
count    2.389270e+05
mean     8.607603e-01
std      3.029878e-01
min      5.235590e-14
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
Name: hwe_pvalue, dtype: float64


In [17]:
for thresh in [1e-3, 1e-4, 1e-5, 1e-6, 1e-10]:
    n_flag = (counts_df["hwe_pvalue"] < thresh).sum()
    print(f"p < {thresh}: {n_flag} probes ({100*n_flag/len(counts_df):.3f}%)")

p < 0.001: 6105 probes (2.555%)
p < 0.0001: 4416 probes (1.848%)
p < 1e-05: 3987 probes (1.669%)
p < 1e-06: 3620 probes (1.515%)
p < 1e-10: 2014 probes (0.843%)


In [18]:
import os

out_dir = r"C:\Users\user\Downloads\GSE148375_clean"
threshold = 1e-6

flagged_probes = counts_df.loc[counts_df["hwe_pvalue"] < threshold, ["probe_id", "hwe_pvalue"]]
print("Number of probes flagged for HWE deviation:", len(flagged_probes))

flagged_probes.to_csv(os.path.join(out_dir, "qc_log_hwe_flagged_probes.csv"), index=False)
print("Saved HWE flag log (no probes dropped from data).")

Number of probes flagged for HWE deviation: 3620
Saved HWE flag log (no probes dropped from data).


In [19]:
import pandas as pd
import numpy as np

geno_path = r"C:\Users\user\Downloads\GSE148375_clean\checkpoint3_snp_probe_filtered.txt"

chunksize = 20000
total_nc = 0
total_cells = 0

reader = pd.read_csv(geno_path, sep="\t", chunksize=chunksize)

for chunk in reader:
    geno_cols = chunk.columns[1:]
    total_nc += (chunk[geno_cols] == "NC").sum().sum()
    total_cells += chunk[geno_cols].size

overall_missing_rate = total_nc / total_cells
print("Total NC cells:", total_nc)
print("Total cells:", total_cells)
print("Overall missing rate:", overall_missing_rate)

Total NC cells: 418408
Total cells: 799927596
Overall missing rate: 0.0005230573393044937


In [20]:
import pandas as pd
import numpy as np
import os

geno_path = r"C:\Users\user\Downloads\GSE148375_clean\checkpoint3_snp_probe_filtered.txt"
out_dir = r"C:\Users\user\Downloads\GSE148375_clean"

chunksize = 20000
out_path = os.path.join(out_dir, "checkpoint6_snp_imputed.txt")
first_chunk = True

reader = pd.read_csv(geno_path, sep="\t", chunksize=chunksize)

for chunk in reader:
    id_col = chunk.columns[0]
    geno_cols = chunk.columns[1:]

    arr = chunk[geno_cols].values

    for i in range(arr.shape[0]):
        row = arr[i]
        nc_mask = (row == "NC")
        if nc_mask.any():
            valid = row[~nc_mask]
            if len(valid) == 0:
                continue  # entire row missing (shouldn't happen post-QC), skip
            vals, counts = np.unique(valid, return_counts=True)
            mode_val = vals[np.argmax(counts)]
            row[nc_mask] = mode_val

    chunk[geno_cols] = arr
    chunk.to_csv(out_path, sep="\t", mode="w" if first_chunk else "a",
                 header=first_chunk, index=False)
    first_chunk = False

print("Saved imputed checkpoint:", out_path)

Saved imputed checkpoint: C:\Users\user\Downloads\GSE148375_clean\checkpoint6_snp_imputed.txt


In [21]:
out_path = r"C:\Users\user\Downloads\GSE148375_clean\checkpoint6_snp_imputed.txt"

chunksize = 20000
total_nc_remaining = 0
total_rows = 0
n_cols = None

reader = pd.read_csv(out_path, sep="\t", chunksize=chunksize)
for chunk in reader:
    geno_cols = chunk.columns[1:]
    if n_cols is None:
        n_cols = len(chunk.columns)
    total_nc_remaining += (chunk[geno_cols] == "NC").sum().sum()
    total_rows += len(chunk)

print("Total rows:", total_rows)
print("Total cols:", n_cols)
print("Remaining NC count:", total_nc_remaining)

Total rows: 238927
Total cols: 3349
Remaining NC count: 0


In [25]:
import os

file_path = r"C:\Users\user\Downloads\HumanExome-12-v1-0-B.csv"

print("Size (MB):", os.path.getsize(file_path) / (1024*1024))

# peek at first ~20 lines raw, since Illumina manifest CSVs often have a metadata header block before the real table
with open(file_path, encoding="utf-8", errors="replace") as f:
    for i, line in enumerate(f):
        print(line.strip())
        if i > 20:
            break

Size (MB): 101.92030620574951
Illumina, Inc.
[Heading]
Descriptor File Name,HumanExome-12v1_B.bpm
Assay Format,Infinium HD Ultra
Date Manufactured,4/22/2014
Loci Count ,247870
[Assay]
IlmnID,Name,IlmnStrand,SNP,AddressA_ID,AlleleA_ProbeSeq,AddressB_ID,AlleleB_ProbeSeq,GenomeBuild,Chr,MapInfo,Ploidy,Species,Source,SourceVersion,SourceStrand,SourceSeq,TopGenomicSeq,BeadSetID,Exp_Clusters,RefStrand
exm-IND1-200449980-0_M_R_1990486447,exm-IND1-200449980,MINUS,[D/I],0091721246,CTGTGGAACTTTCCGGAGCTCGGCACCCTTCCCCCAAGCCTACCTGGGCG,,,37,1,202183358,diploid,Homo sapiens,1000genomes,0,PLUS,GAGGGCCGCTCAGCGAGGGCGGGACAGAACCTCTCCCGGGCTGGGAGTGCACGGCGCGGT[-/G]CGCCCAGGTAGGCTTGGGGGAAGGGTGCCGAGCTCCGGAAAGTTCCACAGGTCCTCCGCA,GAGGGCCGCTCAGCGAGGGCGGGACAGAACCTCTCCCGGGCTGGGAGTGCACGGCGCGGT[-/G]CGCCCAGGTAGGCTTGGGGGAAGGGTGCCGAGCTCCGGAAAGTTCCACAGGTCCTCCGCA,621,3,-
exm-IND1-201453487-0_M_R_1990486448,exm-IND1-201453487,MINUS,[D/I],0040777952,TCCTGCAACCAGGGCCGATACCCCCTCATCCAGACGCTACGGCAGGAACT,,,37,1,203186865,diploid,H

In [26]:
import pandas as pd

manifest_path = r"C:\Users\user\Downloads\HumanExome-12-v1-0-B.csv"

# the real table starts after the [Assay] line -- skip the header block
manifest_df = pd.read_csv(manifest_path, skiprows=7)  # skip Illumina,Inc + [Heading] block + [Assay] line

print(manifest_df.shape)
print(manifest_df.columns.tolist())
print(manifest_df[["IlmnID", "Chr", "MapInfo", "SourceStrand", "SourceSeq"]].head())

# check strand distribution
print(manifest_df["SourceStrand"].value_counts())

(247894, 21)
['IlmnID', 'Name', 'IlmnStrand', 'SNP', 'AddressA_ID', 'AlleleA_ProbeSeq', 'AddressB_ID', 'AlleleB_ProbeSeq', 'GenomeBuild', 'Chr', 'MapInfo', 'Ploidy', 'Species', 'Source', 'SourceVersion', 'SourceStrand', 'SourceSeq', 'TopGenomicSeq', 'BeadSetID', 'Exp_Clusters', 'RefStrand']
                                 IlmnID Chr      MapInfo SourceStrand  \
0   exm-IND1-200449980-0_M_R_1990486447   1  202183358.0         PLUS   
1   exm-IND1-201453487-0_M_R_1990486448   1  203186865.0         PLUS   
2    exm-IND1-85310248-0_P_F_1990486624   1   85537661.0         PLUS   
3  exm-IND10-102817747-0_P_F_1990486569  10  102827757.0         PLUS   
4   exm-IND10-18329639-0_M_R_1990486442  10   18289633.0         PLUS   

                                           SourceSeq  
0  GAGGGCCGCTCAGCGAGGGCGGGACAGAACCTCTCCCGGGCTGGGA...  
1  CATGGAGCTGTGTTTGTACCCCACCTCCAAATTCCACCACTGGCCC...  
2  CTTTATGCAGCCCAAGTCAGAACCATGGCTCCAAAACAAAAGAAAA...  
3  GGGAACACCTGGCTCTGCAAGTTTTGTGCCTTAGCGGAAAGAGGCG

C:\Users\user\AppData\Local\Temp\ipykernel_23144\1012923940.py:6: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  manifest_df = pd.read_csv(manifest_path, skiprows=7)  # skip Illumina,Inc + [Heading] block + [Assay] line


In [28]:
import re
import numpy as np

def extract_alleles(seq):
    if not isinstance(seq, str):
        return None, None
    match = re.search(r"\[([ACGT-]+)/([ACGT-]+)\]", seq)
    if match:
        return match.group(1), match.group(2)
    return None, None

manifest_df["allele1"], manifest_df["allele2"] = zip(*manifest_df["SourceSeq"].map(extract_alleles))

print("Rows with missing SourceSeq:", manifest_df["SourceSeq"].isna().sum())
print(manifest_df[["IlmnID", "SourceStrand", "allele1", "allele2"]].head(10))

indel_like = manifest_df[(manifest_df["allele1"] == "-") | (manifest_df["allele2"] == "-")]
print("Indel-like rows in manifest:", len(indel_like))

failed_extraction = manifest_df[manifest_df["allele1"].isna()]
print("Rows where allele extraction failed:", len(failed_extraction))

Rows with missing SourceSeq: 24
                                 IlmnID SourceStrand allele1 allele2
0   exm-IND1-200449980-0_M_R_1990486447         PLUS       -       G
1   exm-IND1-201453487-0_M_R_1990486448         PLUS       -    ACTC
2    exm-IND1-85310248-0_P_F_1990486624         PLUS       -       A
3  exm-IND10-102817747-0_P_F_1990486569         PLUS       -       T
4   exm-IND10-18329639-0_M_R_1990486442         PLUS       -       G
5   exm-IND10-27476467-0_M_R_1990486443         PLUS       -      GC
6   exm-IND10-27727540-0_M_R_1990486444         PLUS       -       T
7   exm-IND10-74560589-0_P_F_1990486573         PLUS       -       C
8  exm-IND11-102094357-0_P_F_1990486575         PLUS       -      AG
9   exm-IND11-60867390-0_M_R_1990486445         PLUS       -       G
Indel-like rows in manifest: 140
Rows where allele extraction failed: 24


In [29]:
print(manifest_df["RefStrand"].value_counts())

# cross-tab SourceStrand vs RefStrand to understand the relationship
print(pd.crosstab(manifest_df["SourceStrand"], manifest_df["RefStrand"]))

RefStrand
-    127498
+    120372
Name: count, dtype: int64
RefStrand         +      -
SourceStrand              
BOT           60017  62585
MINUS             0      1
PLUS             69     70
TOP           60286  64842


In [30]:
complement_map = {"A": "T", "T": "A", "C": "G", "G": "C"}

def complement_allele(a):
    if not isinstance(a, str) or a == "-" or len(a) != 1:
        return a  # leave multi-base indels or missing as-is, not relevant for SNP-only set
    return complement_map.get(a, a)

def get_forward_strand_allele(allele, ref_strand):
    if ref_strand == "+":
        return allele
    elif ref_strand == "-":
        return complement_allele(allele)
    return None

manifest_df["allele1_fwd"] = [
    get_forward_strand_allele(a, rs) for a, rs in zip(manifest_df["allele1"], manifest_df["RefStrand"])
]
manifest_df["allele2_fwd"] = [
    get_forward_strand_allele(a, rs) for a, rs in zip(manifest_df["allele2"], manifest_df["RefStrand"])
]

print(manifest_df[["IlmnID", "RefStrand", "allele1", "allele2", "allele1_fwd", "allele2_fwd"]].head(10))

                                 IlmnID RefStrand allele1 allele2 allele1_fwd  \
0   exm-IND1-200449980-0_M_R_1990486447         -       -       G           -   
1   exm-IND1-201453487-0_M_R_1990486448         -       -    ACTC           -   
2    exm-IND1-85310248-0_P_F_1990486624         +       -       A           -   
3  exm-IND10-102817747-0_P_F_1990486569         +       -       T           -   
4   exm-IND10-18329639-0_M_R_1990486442         -       -       G           -   
5   exm-IND10-27476467-0_M_R_1990486443         -       -      GC           -   
6   exm-IND10-27727540-0_M_R_1990486444         -       -       T           -   
7   exm-IND10-74560589-0_P_F_1990486573         +       -       C           -   
8  exm-IND11-102094357-0_P_F_1990486575         +       -      AG           -   
9   exm-IND11-60867390-0_M_R_1990486445         -       -       G           -   

  allele2_fwd  
0           C  
1        ACTC  
2           A  
3           T  
4           C  
5          G

In [31]:
is_snp_row = (
    manifest_df["allele1"].str.len() == 1
) & (
    manifest_df["allele2"].str.len() == 1
) & (manifest_df["allele1"] != "-") & (manifest_df["allele2"] != "-")

manifest_snp = manifest_df[is_snp_row].copy()
print("SNP rows in manifest:", len(manifest_snp))

# reference allele = allele1_fwd, by 1000genomes REF/ALT convention (first listed = REF)
manifest_snp = manifest_snp.rename(columns={"allele1_fwd": "ref_allele", "allele2_fwd": "alt_allele"})

print(manifest_snp[["IlmnID", "Chr", "MapInfo", "ref_allele", "alt_allele"]].head(10))

SNP rows in manifest: 247730
                                IlmnID Chr      MapInfo ref_allele alt_allele
137   exm-rs1000005-131_T_F_1990486739  21   34433051.0          G          C
138   exm-rs1000026-131_T_F_1990486741  21   38934599.0          T          C
139   exm-rs1000053-131_T_F_1990486743   2   12790328.0          T          C
140   exm-rs1000110-131_T_F_1990486745   9  117908721.0          T          C
141   exm-rs1000113-131_B_F_1990477628   5  150240076.0          T          C
142   exm-rs1000158-131_T_F_1990486747  20   36599904.0          A          G
143   exm-rs1000192-131_B_F_1990477632  16    6747139.0          A          G
144   exm-rs1000203-131_T_R_1990486750  14   40896108.0          A          G
145  exm-rs10005603-131_T_R_1990486752   4  105844273.0          A          C
146   exm-rs1000797-131_T_R_1990486754  10  129652787.0          T          C


In [32]:
geno_path = r"C:\Users\user\Downloads\GSE148375_clean\checkpoint6_snp_imputed.txt"

with open(geno_path, encoding="utf-8") as f:
    geno_header = f.readline()
    geno_probe_ids = []
    for line in f:
        geno_probe_ids.append(line.split("\t", 1)[0])

geno_probe_set = set(geno_probe_ids)
manifest_probe_set = set(manifest_snp["IlmnID"])

print("Genotype probes:", len(geno_probe_set))
print("Manifest SNP probes:", len(manifest_probe_set))
print("Overlap:", len(geno_probe_set & manifest_probe_set))
print("In genotype but not manifest:", len(geno_probe_set - manifest_probe_set))
print("In manifest but not genotype:", len(manifest_probe_set - geno_probe_set))

Genotype probes: 238927
Manifest SNP probes: 247730
Overlap: 222783
In genotype but not manifest: 16144
In manifest but not genotype: 24947


In [33]:
missing_from_manifest = geno_probe_set - manifest_probe_set
print("Sample of genotype probes missing from manifest_snp:", list(missing_from_manifest)[:10])

# check if they're present in the FULL manifest (before SNP filtering) -- maybe they got excluded by our is_snp_row filter
full_manifest_ids = set(manifest_df["IlmnID"])
still_missing = missing_from_manifest - full_manifest_ids
print("Missing from FULL manifest (not just SNP subset):", len(still_missing))

# of the ones that ARE in the full manifest but got excluded from manifest_snp, check why
recoverable = missing_from_manifest & full_manifest_ids
print("In full manifest but excluded by SNP filter:", len(recoverable))

sample_recoverable = manifest_df[manifest_df["IlmnID"].isin(list(recoverable)[:10])]
print(sample_recoverable[["IlmnID", "SourceStrand", "RefStrand", "allele1", "allele2", "SourceSeq"]])

Sample of genotype probes missing from manifest_snp: ['exm1897053-0_B_R_2060135003', 'exm1341357-0_B_F_2060144843', 'exm766218-0_B_R_2058869863', 'exm458412-0_B_R_2058868953', 'exm339341-0_B_R_2060133580', 'exm2081170-0_T_F_2060140937', 'exm860941-0_T_R_2058864326', 'exm2258009-0_T_F_2060136641', 'exm1352733-0_B_R_2060144328', 'exm287834-0_T_R_2058864689']
Missing from FULL manifest (not just SNP subset): 16144
In full manifest but excluded by SNP filter: 0
Empty DataFrame
Columns: [IlmnID, SourceStrand, RefStrand, allele1, allele2, SourceSeq]
Index: []


In [34]:
# strip trailing _<digits> address suffix, compare core probe names
import re

def strip_address_suffix(probe_id):
    return re.sub(r'_\d+$', '', probe_id)

missing_core_names = {strip_address_suffix(p) for p in missing_from_manifest}
manifest_core_names = {strip_address_suffix(p) for p in full_manifest_ids}

recovered_by_core_name = missing_core_names & manifest_core_names
print("Missing probes recoverable by core-name match:", len(recovered_by_core_name))
print("Still completely missing:", len(missing_core_names - manifest_core_names))

# sample a few to inspect manually
sample_missing_core = list(missing_core_names - manifest_core_names)[:10]
print(sample_missing_core)

Missing probes recoverable by core-name match: 16144
Still completely missing: 0
[]


In [35]:
manifest_df["core_name"] = manifest_df["IlmnID"].map(strip_address_suffix)

dup_counts = manifest_df["core_name"].value_counts()
n_duplicated_core_names = (dup_counts > 1).sum()
print("Core names appearing more than once in manifest:", n_duplicated_core_names)
print(dup_counts[dup_counts > 1].head(10))

Core names appearing more than once in manifest: 0
Series([], Name: count, dtype: int64)


In [36]:
import pandas as pd

# rebuild manifest_snp with core_name included
manifest_df["core_name"] = manifest_df["IlmnID"].map(strip_address_suffix)
manifest_snp = manifest_df[is_snp_row].copy()
manifest_snp = manifest_snp.rename(columns={"allele1_fwd": "ref_allele", "allele2_fwd": "alt_allele"})

# build two lookup dicts: by full IlmnID, and by core_name (for fallback)
exact_lookup = manifest_snp.set_index("IlmnID")[["ref_allele", "alt_allele", "Chr", "MapInfo"]]
core_lookup = manifest_snp.set_index("core_name")[["ref_allele", "alt_allele", "Chr", "MapInfo"]]

def get_ref_info(probe_id):
    if probe_id in exact_lookup.index:
        row = exact_lookup.loc[probe_id]
    else:
        core = strip_address_suffix(probe_id)
        if core in core_lookup.index:
            row = core_lookup.loc[core]
        else:
            return None, None, None, None
    return row["ref_allele"], row["alt_allele"], row["Chr"], row["MapInfo"]

# apply to all genotype probe IDs
ref_info = [get_ref_info(pid) for pid in geno_probe_ids]
ref_df = pd.DataFrame(ref_info, columns=["ref_allele", "alt_allele", "Chr", "MapInfo"])
ref_df["probe_id"] = geno_probe_ids

print("Total genotype probes:", len(ref_df))
print("Probes with resolved ref_allele:", ref_df["ref_allele"].notna().sum())
print("Still unresolved:", ref_df["ref_allele"].isna().sum())

out_dir = r"C:\Users\user\Downloads\GSE148375_clean"
ref_df.to_csv(os.path.join(out_dir, "probe_reference_alleles.csv"), index=False)
print("Saved reference allele lookup table.")

Total genotype probes: 238927
Probes with resolved ref_allele: 238927
Still unresolved: 0
Saved reference allele lookup table.


In [37]:
import pandas as pd
import numpy as np
import os

geno_path = r"C:\Users\user\Downloads\GSE148375_clean\checkpoint6_snp_imputed.txt"
out_dir = r"C:\Users\user\Downloads\GSE148375_clean"

ref_df = pd.read_csv(os.path.join(out_dir, "probe_reference_alleles.csv"))
ref_lookup = ref_df.set_index("probe_id")["ref_allele"]

chunksize = 20000
encoded_chunks = []
probe_order = []

reader = pd.read_csv(geno_path, sep="\t", chunksize=chunksize)

for chunk in reader:
    id_col = chunk.columns[0]
    geno_cols = chunk.columns[1:]
    ids = chunk[id_col].values
    probe_order.extend(ids.tolist())

    ref_alleles = ref_lookup.loc[ids].values  # aligned per row

    arr = chunk[geno_cols].values.astype('<U2')
    arr = np.ascontiguousarray(arr)
    chars = arr.view('U1').reshape(arr.shape[0], arr.shape[1], 2)
    first, second = chars[:, :, 0], chars[:, :, 1]

    ref_col = ref_alleles[:, None]  # broadcast per row

    match_first = (first == ref_col)
    match_second = (second == ref_col)
    n_ref_matches = match_first.astype(np.int8) + match_second.astype(np.int8)

    # n_ref_matches: 2 -> hom ref -> encode 0 ; 1 -> het -> encode 1 ; 0 -> hom alt -> encode 2
    encoded = 2 - n_ref_matches  # 2->0, 1->1, 0->2

    encoded_chunks.append(encoded.astype(np.int8))

encoded_matrix = np.vstack(encoded_chunks)
print("Encoded matrix shape:", encoded_matrix.shape)
print("Value counts overall:", np.unique(encoded_matrix, return_counts=True))

C:\Users\user\AppData\Local\Temp\ipykernel_23144\340909817.py:8: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  ref_df = pd.read_csv(os.path.join(out_dir, "probe_reference_alleles.csv"))


Encoded matrix shape: (238927, 3348)
Value counts overall: (array([0, 1, 2], dtype=int8), array([143046689,  20451609, 636429298], dtype=int64))


In [38]:
# pick a probe we know is monomorphic from way back (exm41, all 'GG')
test_probe = "exm41-0_B_F_1921435147"

print("Reference allele for this probe:", ref_lookup.loc[test_probe])

# find its row index in our chunk processing to check actual encoded values
idx = probe_order.index(test_probe)
print("Encoded values for this probe (should be all same since monomorphic):")
print(np.unique(encoded_matrix[idx], return_counts=True))

Reference allele for this probe: T
Encoded values for this probe (should be all same since monomorphic):
(array([2], dtype=int8), array([3348], dtype=int64))


In [39]:
test_row = manifest_df[manifest_df["IlmnID"] == test_probe]
print(test_row[["IlmnID", "SourceStrand", "RefStrand", "allele1", "allele2", "allele1_fwd", "allele2_fwd", "SourceSeq"]].to_string())

                        IlmnID SourceStrand RefStrand allele1 allele2 allele1_fwd allele2_fwd                                                                                                                      SourceSeq
162290  exm41-0_B_F_1921435147          BOT         +       T       C           T           C  GCTCTGCTCGCAGGGAAAAGTCTGAAGACGCTTATGTCCAAGGGGATCCTGCAGGTGCAT[T/C]CTCCGATCTGCGACTGCCCGGGCTGCCGAATATCCTCCCCGGTGGTGAGATGCGGGGCTC


In [40]:
# merge ref_df with manifest strand info to check for systematic bias by RefStrand
ref_df_check = ref_df.merge(
    manifest_df[["IlmnID", "RefStrand", "SourceStrand"]],
    left_on="probe_id", right_on="IlmnID", how="left"
)

# compute mean encoded value per probe, grouped by RefStrand
mean_encoded_per_probe = encoded_matrix.mean(axis=1)
ref_df_check["mean_encoded"] = mean_encoded_per_probe

print(ref_df_check.groupby("RefStrand")["mean_encoded"].describe())

              count      mean       std  min       25%  50%  75%  max
RefStrand                                                            
+          108209.0  1.631562  0.739056  0.0  1.975508  2.0  2.0  2.0
-          114574.0  1.612034  0.758819  0.0  1.971027  2.0  2.0  2.0


In [41]:
print(manifest_df[manifest_df["IlmnID"] == test_probe][["IlmnID", "IlmnStrand", "SourceStrand", "RefStrand"]])

# check overall: does IlmnStrand match SourceStrand consistently, or do they diverge?
print(pd.crosstab(manifest_df["IlmnStrand"], manifest_df["SourceStrand"]))

                        IlmnID IlmnStrand SourceStrand RefStrand
162290  exm41-0_B_F_1921435147        BOT          BOT         +
SourceStrand    BOT  MINUS  PLUS    TOP
IlmnStrand                             
BOT           59978      0     0  64882
MINUS             0      0    70      0
PLUS              0      1    69      0
TOP           62624      0     0  60246


In [42]:
test_row_full = manifest_df[manifest_df["IlmnID"] == test_probe]
print(test_row_full[["IlmnID", "IlmnStrand", "SourceStrand", "RefStrand", "SourceSeq", "TopGenomicSeq"]].to_string())

# also extract the bracketed allele pair from TopGenomicSeq directly
import re
seq = test_row_full["TopGenomicSeq"].values[0]
match = re.search(r"\[([ACGT-]+)/([ACGT-]+)\]", seq)
print("TopGenomicSeq allele pair:", match.groups() if match else None)

                        IlmnID IlmnStrand SourceStrand RefStrand                                                                                                                      SourceSeq                                                                                                                  TopGenomicSeq
162290  exm41-0_B_F_1921435147        BOT          BOT         +  GCTCTGCTCGCAGGGAAAAGTCTGAAGACGCTTATGTCCAAGGGGATCCTGCAGGTGCAT[T/C]CTCCGATCTGCGACTGCCCGGGCTGCCGAATATCCTCCCCGGTGGTGAGATGCGGGGCTC  GAGCCCCGCATCTCACCACCGGGGAGGATATTCGGCAGCCCGGGCAGTCGCAGATCGGAG[A/G]ATGCACCTGCAGGATCCCCTTGGACATAAGCGTCTTCAGACTTTTCCCTGCGAGCAGAGC
TopGenomicSeq allele pair: ('A', 'G')


In [43]:
manifest_df["flip1"] = manifest_df["IlmnStrand"] == "BOT"
manifest_df["flip2"] = manifest_df["RefStrand"] == "-"
manifest_df["total_flip"] = manifest_df["flip1"] != manifest_df["flip2"]

print(manifest_df["total_flip"].value_counts())

total_flip
False    125228
True     122666
Name: count, dtype: int64


In [44]:
import pandas as pd
import numpy as np
import os

geno_path = r"C:\Users\user\Downloads\GSE148375_clean\checkpoint6_snp_imputed.txt"
out_dir = r"C:\Users\user\Downloads\GSE148375_clean"

# rebuild a clean per-probe lookup: ref_allele (plus-strand) + whether raw letters need complementing
manifest_df["flip1"] = manifest_df["IlmnStrand"] == "BOT"
manifest_df["flip2"] = manifest_df["RefStrand"] == "-"
manifest_df["total_flip"] = manifest_df["flip1"] != manifest_df["flip2"]

manifest_snp = manifest_df[is_snp_row].copy()
manifest_snp = manifest_snp.rename(columns={"allele1_fwd": "ref_allele", "allele2_fwd": "alt_allele"})
manifest_snp["core_name"] = manifest_snp["IlmnID"].map(strip_address_suffix)

exact_lookup2 = manifest_snp.set_index("IlmnID")[["ref_allele", "alt_allele", "total_flip"]]
core_lookup2 = manifest_snp.set_index("core_name")[["ref_allele", "alt_allele", "total_flip"]]

def get_ref_info2(probe_id):
    if probe_id in exact_lookup2.index:
        row = exact_lookup2.loc[probe_id]
    else:
        core = strip_address_suffix(probe_id)
        if core in core_lookup2.index:
            row = core_lookup2.loc[core]
        else:
            return None, None, None
    return row["ref_allele"], row["alt_allele"], row["total_flip"]

ref_info2 = [get_ref_info2(pid) for pid in geno_probe_ids]
ref_df2 = pd.DataFrame(ref_info2, columns=["ref_allele", "alt_allele", "total_flip"])
ref_df2["probe_id"] = geno_probe_ids

print("Unresolved after rebuild:", ref_df2["ref_allele"].isna().sum())
ref_df2.to_csv(os.path.join(out_dir, "probe_reference_alleles_v2.csv"), index=False)

Unresolved after rebuild: 0


In [45]:
complement_map = {"A": "T", "T": "A", "C": "G", "G": "C", "N": "N"}
comp_table = np.array([complement_map.get(chr(i), chr(i)) for i in range(256)])

ref_lookup2 = ref_df2.set_index("probe_id")["ref_allele"]
flip_lookup2 = ref_df2.set_index("probe_id")["total_flip"]

chunksize = 20000
encoded_chunks = []
probe_order = []

reader = pd.read_csv(geno_path, sep="\t", chunksize=chunksize)

for chunk in reader:
    id_col = chunk.columns[0]
    geno_cols = chunk.columns[1:]
    ids = chunk[id_col].values
    probe_order.extend(ids.tolist())

    ref_alleles = ref_lookup2.loc[ids].values
    flips = flip_lookup2.loc[ids].values

    arr = chunk[geno_cols].values.astype('<U2')
    arr = np.ascontiguousarray(arr)
    chars = arr.view('U1').reshape(arr.shape[0], arr.shape[1], 2)
    first, second = chars[:, :, 0].copy(), chars[:, :, 1].copy()

    # apply complement where total_flip is True, row-wise
    flip_mask = flips[:, None]  # broadcast across samples
    comp_v = np.vectorize(lambda c: complement_map.get(c, c))

    first_flipped = np.where(flip_mask, comp_v(first), first)
    second_flipped = np.where(flip_mask, comp_v(second), second)

    ref_col = ref_alleles[:, None]
    match_first = (first_flipped == ref_col)
    match_second = (second_flipped == ref_col)
    n_ref_matches = match_first.astype(np.int8) + match_second.astype(np.int8)
    encoded = 2 - n_ref_matches

    encoded_chunks.append(encoded.astype(np.int8))

encoded_matrix = np.vstack(encoded_chunks)
print("Encoded matrix shape:", encoded_matrix.shape)
print("Value counts overall:", np.unique(encoded_matrix, return_counts=True))

Encoded matrix shape: (238927, 3348)
Value counts overall: (array([0, 1, 2], dtype=int8), array([138767362,  20782716, 640377518], dtype=int64))


In [46]:
test_probe = "exm41-0_B_F_1921435147"
idx = probe_order.index(test_probe)

print("ref_allele:", ref_lookup2.loc[test_probe])
print("total_flip:", flip_lookup2.loc[test_probe])
print("Encoded values for this probe:")
print(np.unique(encoded_matrix[idx], return_counts=True))

ref_allele: T
total_flip: True
Encoded values for this probe:
(array([2], dtype=int8), array([3348], dtype=int64))


In [48]:
import re

def get_bracket(seq):
    if not isinstance(seq, str):
        return None, None
    m = re.search(r"\[([ACGT-]+)/([ACGT-]+)\]", seq)
    return m.groups() if m else (None, None)

manifest_df["topgen_allele1"], manifest_df["topgen_allele2"] = zip(*manifest_df["TopGenomicSeq"].map(get_bracket))

# quick check: for IlmnStrand==TOP rows, topgen pair should equal SourceSeq pair (no complement)
top_check = manifest_df[manifest_df["IlmnStrand"] == "TOP"].head(5)
print(top_check[["IlmnID", "allele1", "allele2", "topgen_allele1", "topgen_allele2"]])

                               IlmnID allele1 allele2 topgen_allele1  \
137  exm-rs1000005-131_T_F_1990486739       C       G              C   
138  exm-rs1000026-131_T_F_1990486741       A       G              A   
139  exm-rs1000053-131_T_F_1990486743       A       G              A   
140  exm-rs1000110-131_T_F_1990486745       A       G              A   
142  exm-rs1000158-131_T_F_1990486747       A       G              A   

    topgen_allele2  
137              G  
138              G  
139              G  
140              G  
142              G  


In [50]:
import pandas as pd
import numpy as np
import os

manifest_snp = manifest_df[is_snp_row].copy()
manifest_snp = manifest_snp.rename(columns={"topgen_allele1": "ref_allele", "topgen_allele2": "alt_allele"})
manifest_snp["core_name"] = manifest_snp["IlmnID"].map(strip_address_suffix)

exact_lookup3 = manifest_snp.set_index("IlmnID")["ref_allele"]
core_lookup3 = manifest_snp.set_index("core_name")["ref_allele"]

def get_ref_v3(probe_id):
    if probe_id in exact_lookup3.index:
        return exact_lookup3.loc[probe_id]
    core = strip_address_suffix(probe_id)
    if core in core_lookup3.index:
        return core_lookup3.loc[core]
    return None

ref_v3 = [get_ref_v3(pid) for pid in geno_probe_ids]
ref_df3 = pd.DataFrame({"probe_id": geno_probe_ids, "ref_allele": ref_v3})

print("Unresolved:", ref_df3["ref_allele"].isna().sum())

out_dir = r"C:\Users\user\Downloads\GSE148375_clean"
ref_df3.to_csv(os.path.join(out_dir, "probe_reference_alleles_v3.csv"), index=False)

Unresolved: 0


In [51]:
ref_lookup3 = ref_df3.set_index("probe_id")["ref_allele"]

chunksize = 20000
encoded_chunks = []
probe_order = []

geno_path = r"C:\Users\user\Downloads\GSE148375_clean\checkpoint6_snp_imputed.txt"
reader = pd.read_csv(geno_path, sep="\t", chunksize=chunksize)

for chunk in reader:
    id_col = chunk.columns[0]
    geno_cols = chunk.columns[1:]
    ids = chunk[id_col].values
    probe_order.extend(ids.tolist())

    ref_alleles = ref_lookup3.loc[ids].values

    arr = chunk[geno_cols].values.astype('<U2')
    arr = np.ascontiguousarray(arr)
    chars = arr.view('U1').reshape(arr.shape[0], arr.shape[1], 2)
    first, second = chars[:, :, 0], chars[:, :, 1]

    ref_col = ref_alleles[:, None]
    match_first = (first == ref_col)
    match_second = (second == ref_col)
    n_ref_matches = match_first.astype(np.int8) + match_second.astype(np.int8)
    encoded = 2 - n_ref_matches

    encoded_chunks.append(encoded.astype(np.int8))

encoded_matrix = np.vstack(encoded_chunks)
print("Encoded matrix shape:", encoded_matrix.shape)
print("Value counts overall:", np.unique(encoded_matrix, return_counts=True))

# re-check our test probe
idx = probe_order.index("exm41-0_B_F_1921435147")
print("Test probe encoded:", np.unique(encoded_matrix[idx], return_counts=True))

Encoded matrix shape: (238927, 3348)
Value counts overall: (array([0, 1, 2], dtype=int8), array([235889181,  37740356, 526298059], dtype=int64))
Test probe encoded: (array([2], dtype=int8), array([3348], dtype=int64))


In [52]:
import pandas as pd
import numpy as np
import os

geno_path = r"C:\Users\user\Downloads\GSE148375_clean\checkpoint6_snp_imputed.txt"
out_dir = r"C:\Users\user\Downloads\GSE148375_clean"

chunksize = 20000
major_alleles = []
probe_ids_all = []

reader = pd.read_csv(geno_path, sep="\t", chunksize=chunksize)

for chunk in reader:
    id_col = chunk.columns[0]
    geno_cols = chunk.columns[1:]
    ids = chunk[id_col].values
    probe_ids_all.extend(ids.tolist())

    arr = chunk[geno_cols].values.astype('<U2')
    arr = np.ascontiguousarray(arr)
    chars = arr.view('U1').reshape(arr.shape[0], arr.shape[1], 2)
    first, second = chars[:, :, 0], chars[:, :, 1]

    # count allele occurrences per probe (each genotype contributes 2 allele calls)
    for i in range(arr.shape[0]):
        all_alleles = np.concatenate([first[i], second[i]])
        vals, counts = np.unique(all_alleles, return_counts=True)
        major = vals[np.argmax(counts)]
        major_alleles.append(major)

major_df = pd.DataFrame({"probe_id": probe_ids_all, "major_allele": major_alleles})
print(major_df.shape)
print(major_df.head())

major_df.to_csv(os.path.join(out_dir, "probe_major_alleles.csv"), index=False)
print("Saved major allele lookup.")

(238927, 2)
                      probe_id major_allele
0  exm2268640-0_B_F_1984844585            G
1       exm41-0_B_F_1921435147            G
2  exm1916089-0_B_R_1927689775            G
3       exm44-0_B_R_1921538602            G
4       exm46-0_T_F_1921333919            G
Saved major allele lookup.


In [54]:
import pandas as pd
import numpy as np
import os

geno_path = r"C:\Users\user\Downloads\GSE148375_clean\checkpoint6_snp_imputed.txt"
out_dir = r"C:\Users\user\Downloads\GSE148375_clean"

major_lookup = major_df.set_index("probe_id")["major_allele"]

chunksize = 20000
encoded_chunks = []
probe_order = []

reader = pd.read_csv(geno_path, sep="\t", chunksize=chunksize)

for chunk in reader:
    id_col = chunk.columns[0]
    geno_cols = chunk.columns[1:]
    ids = chunk[id_col].values
    probe_order.extend(ids.tolist())

    ref_alleles = major_lookup.loc[ids].values

    arr = chunk[geno_cols].values.astype('<U2')
    arr = np.ascontiguousarray(arr)
    chars = arr.view('U1').reshape(arr.shape[0], arr.shape[1], 2)
    first, second = chars[:, :, 0], chars[:, :, 1]

    ref_col = ref_alleles[:, None]
    match_first = (first == ref_col)
    match_second = (second == ref_col)
    n_ref_matches = match_first.astype(np.int8) + match_second.astype(np.int8)
    encoded = 2 - n_ref_matches  # 2 matches->0 (hom ref), 1 match->1 (het), 0 matches->2 (hom alt)

    encoded_chunks.append(encoded.astype(np.int8))

encoded_matrix = np.vstack(encoded_chunks)
print("Encoded matrix shape:", encoded_matrix.shape)
print("Value counts overall:", np.unique(encoded_matrix, return_counts=True))

Encoded matrix shape: (238927, 3348)
Value counts overall: (array([0, 1, 2], dtype=int8), array([753189742,  37740356,   8997498], dtype=int64))


In [55]:
import os

out_dir = r"C:\Users\user\Downloads\GSE148375_clean"

# save the encoded matrix with probe IDs, as a checkpoint
encoded_df = pd.DataFrame(encoded_matrix, columns=[c for c in pd.read_csv(geno_path, sep="\t", nrows=0).columns[1:]])
encoded_df.insert(0, "probe_id", probe_order)

print(encoded_df.shape)
print(encoded_df.head())

encoded_df.to_csv(os.path.join(out_dir, "checkpoint7_snp_encoded_012.csv"), index=False)
print("Saved encoded SNP matrix (probes as rows, samples as columns).")

(238927, 3349)
                      probe_id  1900017  1900029  1900030  1900031  1900040  \
0  exm2268640-0_B_F_1984844585        1        0        0        0        0   
1       exm41-0_B_F_1921435147        0        0        0        0        0   
2  exm1916089-0_B_R_1927689775        0        0        0        0        0   
3       exm44-0_B_R_1921538602        0        0        0        0        0   
4       exm46-0_T_F_1921333919        0        0        0        0        0   

   1900043  1900046  1900048  1900049  ...  215373  215606  215762  215763  \
0        0        0        0        0  ...       0       0       0       0   
1        0        0        0        0  ...       0       0       0       0   
2        0        0        0        0  ...       0       0       0       0   
3        0        0        0        0  ...       0       0       0       0   
4        0        0        0        0  ...       0       0       0       0   

   215764  215769  215771  215772  215773

In [1]:
import pandas as pd
import os

out_dir = r"C:\Users\user\Downloads\GSE148375_clean"

encoded_df = pd.read_csv(os.path.join(out_dir, "checkpoint7_snp_encoded_012.csv"))

# transpose: set probe_id as index, transpose, then sample_id becomes the index
encoded_T = encoded_df.set_index("probe_id").T
encoded_T.index.name = "sample_id"
encoded_T = encoded_T.reset_index()

print("Transposed shape:", encoded_T.shape)
print(encoded_T.iloc[:5, :6])

Transposed shape: (3348, 238928)
probe_id sample_id  exm2268640-0_B_F_1984844585  exm41-0_B_F_1921435147  \
0          1900017                            1                       0   
1          1900029                            0                       0   
2          1900030                            0                       0   
3          1900031                            0                       0   
4          1900040                            0                       0   

probe_id  exm1916089-0_B_R_1927689775  exm44-0_B_R_1921538602  \
0                                   0                       0   
1                                   0                       0   
2                                   0                       0   
3                                   0                       0   
4                                   0                       0   

probe_id  exm46-0_T_F_1921333919  
0                              0  
1                              0  
2                   

In [2]:
print(encoded_T.dtypes.value_counts())
print("Estimated memory (MB):", encoded_T.memory_usage(deep=False).sum() / 1e6)

int64     238927
object         1
Name: count, dtype: int64
Estimated memory (MB): 6399.447684


In [3]:
import numpy as np

snp_cols_T = encoded_T.columns[1:]  # exclude sample_id
encoded_T[snp_cols_T] = encoded_T[snp_cols_T].astype(np.int8)

print(encoded_T.dtypes.value_counts())
print("Estimated memory (MB):", encoded_T.memory_usage(deep=False).sum() / 1e6)

int8      238927
object         1
Name: count, dtype: int64
Estimated memory (MB): 799.954512


In [4]:
import pandas as pd
import os

out_dir = r"C:\Users\user\Downloads\GSE148375_clean"

meta_df_final = pd.read_csv(os.path.join(out_dir, "checkpoint2_metadata_sample_filtered.csv"))
meta_df_final["sample_id"] = meta_df_final["sample_id"].astype(str)

print("Metadata samples:", meta_df_final.shape[0])
print("Genotype samples:", encoded_T.shape[0])
print("Overlap:", len(set(meta_df_final["sample_id"]) & set(encoded_T["sample_id"])))

# join via index alignment instead of merge (lighter on memory)
meta_indexed = meta_df_final.set_index("sample_id")
geno_indexed = encoded_T.set_index("sample_id")

final_df = meta_indexed.join(geno_indexed, how="inner")
final_df = final_df.reset_index()

print("Final merged shape:", final_df.shape)
print(final_df.iloc[:5, :12])

Metadata samples: 3348
Genotype samples: 3348
Overlap: 3348
Final merged shape: (3348, 238936)
  sample_id         ethnicity  age  gender  cpd  hsi  ftnd smoking_status  \
0    200026  African-American   39    Male   20    4     7         Smoker   
1    200027  African-American   42    Male   30    5     9         Smoker   
2    200028  African-American   32  Female   40    6     9         Smoker   
3    200032  African-American   33    Male   20    4     7         Smoker   
4    200033  African-American   48  Female   10    3     5         Smoker   

  tissue  exm2268640-0_B_F_1984844585  exm41-0_B_F_1921435147  \
0  Blood                            0                       0   
1  Blood                            0                       0   
2  Blood                            0                       0   
3  Blood                            2                       0   
4  Blood                            1                       0   

   exm1916089-0_B_R_1927689775  
0                 

In [5]:
import os

out_dir = r"C:\Users\user\Downloads\GSE148375_clean"

final_df.to_csv(os.path.join(out_dir, "checkpoint8_final_features_targets.csv"), index=False)
print("Saved final merged dataset:", os.path.join(out_dir, "checkpoint8_final_features_targets.csv"))
print("Shape:", final_df.shape)

Saved final merged dataset: C:\Users\user\Downloads\GSE148375_clean\checkpoint8_final_features_targets.csv
Shape: (3348, 238936)


In [1]:
import pandas as pd
import numpy as np

out_dir = r"C:\Users\user\Downloads\GSE148375_clean"

encoded_df = pd.read_csv(out_dir + r"\checkpoint7_snp_encoded_012.csv")
print("Loaded shape:", encoded_df.shape)

probe_ids = encoded_df["probe_id"].values
sample_cols = encoded_df.columns[1:]

X_snp_first = encoded_df[sample_cols].to_numpy(dtype=np.int8)
print("X_snp_first shape (SNPs x samples):", X_snp_first.shape)

X = X_snp_first.T
print("X shape (samples x SNPs):", X.shape)
print("Memory (MB):", X.nbytes / 1e6)

sample_ids = sample_cols.tolist()

Loaded shape: (238927, 3349)
X_snp_first shape (SNPs x samples): (238927, 3348)
X shape (samples x SNPs): (3348, 238927)
Memory (MB): 799.927596


In [2]:
from sklearn.decomposition import PCA
import numpy as np

# PCA expects float input; convert (memory will roughly double temporarily during this step)
X_float = X.astype(np.float32)

pca = PCA(n_components=10, random_state=42)
pcs = pca.fit_transform(X_float)

print("PC scores shape:", pcs.shape)
print("Explained variance ratio per PC:")
print(pca.explained_variance_ratio_)
print("Cumulative variance explained:", np.cumsum(pca.explained_variance_ratio_))

PC scores shape: (3348, 10)
Explained variance ratio per PC:
[0.01791282 0.0030218  0.0027829  0.00236773 0.00194487 0.00178041
 0.00174893 0.00155811 0.00149052 0.00145239]
Cumulative variance explained: [0.01791282 0.02093462 0.02371752 0.02608525 0.02803012 0.02981053
 0.03155946 0.03311757 0.03460809 0.03606047]


In [3]:
import pandas as pd

meta_df_check = pd.read_csv(r"C:\Users\user\Downloads\GSE148375_clean\checkpoint2_metadata_sample_filtered.csv")
meta_df_check["sample_id"] = meta_df_check["sample_id"].astype(str)

pc_df = pd.DataFrame(pcs, columns=[f"PC{i+1}" for i in range(10)])
pc_df["sample_id"] = sample_ids
pc_df["sample_id"] = pc_df["sample_id"].astype(str)

merged_check = meta_df_check.merge(pc_df, on="sample_id", how="inner")

# correlation of PC1 with age, and group means by gender/smoking_status
print(merged_check[["PC1", "age"]].corr())
print(merged_check.groupby("gender")["PC1"].mean())
print(merged_check.groupby("smoking_status")["PC1"].mean())

          PC1       age
PC1  1.000000  0.001975
age  0.001975  1.000000
gender
Female   -0.665653
Male      0.760198
Name: PC1, dtype: float32
smoking_status
Non-smoker   -0.181017
Smoker        0.190561
Name: PC1, dtype: float32


In [4]:
import pandas as pd

manifest_path = r"C:\Users\user\Downloads\HumanExome-12-v1-0-B.csv"
manifest_df = pd.read_csv(manifest_path, skiprows=7, low_memory=False)

# check what chromosome labels actually appear
print(manifest_df["Chr"].value_counts())

Chr
1     25178
2     17542
11    15894
6     15507
19    15254
3     14805
17    13280
12    12513
5     11561
7     11347
16    10577
4     10408
9     10402
10     9675
8      9101
15     8377
14     7867
20     6537
22     5231
X      5104
13     4446
18     3812
21     2873
MT      215
Y       135
0       122
XY      107
Name: count, dtype: int64


In [5]:
non_autosomal = {"X", "Y", "XY", "MT", "0"}

sex_linked_probes = set(manifest_df.loc[manifest_df["Chr"].isin(non_autosomal), "IlmnID"])
print("Number of non-autosomal probes in manifest:", len(sex_linked_probes))

# also account for the address-suffix versioning mismatch we handled earlier
def strip_address_suffix(probe_id):
    import re
    return re.sub(r'_\d+$', '', probe_id)

sex_linked_core_names = {strip_address_suffix(p) for p in sex_linked_probes}

# check overlap with our actual probe set (probe_ids from checkpoint7)
our_probe_core_names = {strip_address_suffix(p): p for p in probe_ids}  # map core->full
to_exclude = set()
for core in sex_linked_core_names:
    if core in our_probe_core_names:
        to_exclude.add(our_probe_core_names[core])

print("Number of our probes to exclude (sex-linked/unmapped):", len(to_exclude))

Number of non-autosomal probes in manifest: 5683
Number of our probes to exclude (sex-linked/unmapped): 5317


In [7]:
import numpy as np

# work directly with numpy arrays — avoid pandas' row filtering overhead entirely
probe_id_array = encoded_df["probe_id"].to_numpy()
keep_mask = ~np.isin(probe_id_array, list(to_exclude))

print("Probes before:", len(probe_id_array))
print("Probes to keep:", keep_mask.sum())

# X_snp_first was already built earlier as int8 numpy array (SNPs x samples)
X_snp_first_auto = X_snp_first[keep_mask]  # filter rows directly in numpy, no pandas copy
X_auto = X_snp_first_auto.T

print("X_auto shape (samples x SNPs):", X_auto.shape)
print("Memory (MB):", X_auto.nbytes / 1e6)

Probes before: 238927
Probes to keep: 233610
X_auto shape (samples x SNPs): (3348, 233610)
Memory (MB): 782.12628


In [9]:
from sklearn.decomposition import PCA
import numpy as np
import gc

# free memory from objects we don't need anymore
del X, X_snp_first, X_snp_first_auto
gc.collect()

X_auto_float = X_auto.astype(np.float32)
del X_auto
gc.collect()

pca_auto = PCA(n_components=10, random_state=42, copy=False, svd_solver='randomized')
pcs_auto = pca_auto.fit_transform(X_auto_float)

print("PC scores shape:", pcs_auto.shape)
print("Explained variance ratio per PC:")
print(pca_auto.explained_variance_ratio_)
print("Cumulative variance explained:", np.cumsum(pca_auto.explained_variance_ratio_))

PC scores shape: (3348, 10)
Explained variance ratio per PC:
[0.01784274 0.00311701 0.00287145 0.00244002 0.00200164 0.00183018
 0.00179899 0.0016026  0.00152336 0.00149069]
Cumulative variance explained: [0.01784274 0.02095976 0.02383121 0.02627122 0.02827287 0.03010304
 0.03190203 0.03350462 0.03502798 0.03651867]


In [10]:
import pandas as pd

meta_df_check = pd.read_csv(r"C:\Users\user\Downloads\GSE148375_clean\checkpoint2_metadata_sample_filtered.csv")
meta_df_check["sample_id"] = meta_df_check["sample_id"].astype(str)

pc_df_auto = pd.DataFrame(pcs_auto, columns=[f"PC{i+1}" for i in range(10)])
pc_df_auto["sample_id"] = sample_ids
pc_df_auto["sample_id"] = pc_df_auto["sample_id"].astype(str)

merged_check_auto = meta_df_check.merge(pc_df_auto, on="sample_id", how="inner")

print(merged_check_auto[["PC1", "age"]].corr())
print(merged_check_auto.groupby("gender")["PC1"].mean())
print(merged_check_auto.groupby("smoking_status")["PC1"].mean())

          PC1       age
PC1  1.000000  0.002916
age  0.002916  1.000000
gender
Female   -0.651453
Male      0.743982
Name: PC1, dtype: float32
smoking_status
Non-smoker   -0.170938
Smoker        0.179951
Name: PC1, dtype: float32


In [11]:
import numpy as np

# we need gender as a 0/1 vector aligned to sample_ids order
gender_map = meta_df_check.set_index("sample_id")["gender"]
gender_aligned = np.array([gender_map.get(sid, None) for sid in sample_ids])
print("Unmatched:", pd.isna(gender_aligned).sum())

gender_binary = (gender_aligned == "Male").astype(int)

# we need X_auto_float still in memory (samples x SNPs) -- if deleted, reload from X_snp_first_auto equivalent
# quick correlation of each SNP with gender (vectorized)
# standardize approach: compute correlation via covariance / std
X_centered = X_auto_float - X_auto_float.mean(axis=0)
gender_centered = gender_binary - gender_binary.mean()

cov = (X_centered * gender_centered[:, None]).mean(axis=0)
snp_std = X_auto_float.std(axis=0)
gender_std = gender_binary.std()

corr_with_gender = cov / (snp_std * gender_std + 1e-9)

print("Number of SNPs with |corr| > 0.3 with gender:", (np.abs(corr_with_gender) > 0.3).sum())
print("Number of SNPs with |corr| > 0.5 with gender:", (np.abs(corr_with_gender) > 0.5).sum())
print("Max correlation:", np.abs(corr_with_gender).max())

Unmatched: 0
Number of SNPs with |corr| > 0.3 with gender: 2
Number of SNPs with |corr| > 0.5 with gender: 1
Max correlation: 0.8597269556364392


In [12]:
strong_idx = np.where(np.abs(corr_with_gender) > 0.3)[0]
strong_probe_ids = probe_id_array[keep_mask][strong_idx]

print("Probe IDs strongly correlated with gender:", strong_probe_ids)
print("Correlation values:", corr_with_gender[strong_idx])

# check their manifest chromosome label
for pid in strong_probe_ids:
    core = strip_address_suffix(pid)
    match = manifest_df[manifest_df["IlmnID"].apply(strip_address_suffix) == core]
    print(pid, "-> Chr:", match["Chr"].values if len(match) else "NOT FOUND IN MANIFEST")

Probe IDs strongly correlated with gender: ['exm2277017-0_T_R_1989215336' 'exm294480-0_B_F_1922170143']
Correlation values: [0.85972696 0.4709405 ]
exm2277017-0_T_R_1989215336 -> Chr: <ArrowStringArray>
['1']
Length: 1, dtype: str
exm294480-0_B_F_1922170143 -> Chr: <ArrowStringArray>
['3']
Length: 1, dtype: str


In [13]:
import numpy as np

# X_auto_float is samples x SNPs, still in memory from before
n_samples = X_auto_float.shape[0]

# allele frequency p per SNP: mean genotype / 2 (since genotype in {0,1,2})
p = X_auto_float.mean(axis=0) / 2

# standard deviation under Hardy-Weinberg: sqrt(2p(1-p))
denom = np.sqrt(2 * p * (1 - p))

# avoid division by zero for monomorphic SNPs (p=0 or p=1)
valid_snp_mask = denom > 1e-8
print("Monomorphic/invalid SNPs to exclude:", (~valid_snp_mask).sum())

X_valid = X_auto_float[:, valid_snp_mask]
p_valid = p[valid_snp_mask]
denom_valid = denom[valid_snp_mask]

X_standardized = (X_valid - 2 * p_valid) / denom_valid

print("Standardized matrix shape:", X_standardized.shape)
print("Mean (should be ~0):", X_standardized.mean())
print("Std (should be ~1):", X_standardized.std())

Monomorphic/invalid SNPs to exclude: 91965
Standardized matrix shape: (3348, 141645)
Mean (should be ~0): 4.820261e-10
Std (should be ~1): 1.0013081


In [14]:
from sklearn.decomposition import PCA
import numpy as np
import gc

del X_auto_float, X_valid
gc.collect()

pca_std = PCA(n_components=10, random_state=42, copy=False, svd_solver='randomized')
pcs_std = pca_std.fit_transform(X_standardized)

print("PC scores shape:", pcs_std.shape)
print("Explained variance ratio per PC:")
print(pca_std.explained_variance_ratio_)
print("Cumulative variance explained:", np.cumsum(pca_std.explained_variance_ratio_))

PC scores shape: (3348, 10)
Explained variance ratio per PC:
[0.00508867 0.00135531 0.00108169 0.00099204 0.00093676 0.00092401
 0.00091995 0.00089464 0.00087454 0.00087181]
Cumulative variance explained: [0.00508867 0.00644397 0.00752566 0.0085177  0.00945446 0.01037847
 0.01129842 0.01219305 0.0130676  0.01393941]


In [15]:
pc_df_std = pd.DataFrame(pcs_std, columns=[f"PC{i+1}" for i in range(10)])
pc_df_std["sample_id"] = sample_ids
pc_df_std["sample_id"] = pc_df_std["sample_id"].astype(str)

merged_check_std = meta_df_check.merge(pc_df_std, on="sample_id", how="inner")

print(merged_check_std[["PC1", "age"]].corr())
print(merged_check_std.groupby("gender")["PC1"].mean())
print(merged_check_std.groupby("smoking_status")["PC1"].mean())

          PC1       age
PC1  1.000000  0.003753
age  0.003753  1.000000
gender
Female   -1.098441
Male      1.254457
Name: PC1, dtype: float32
smoking_status
Non-smoker   -0.443883
Smoker        0.467288
Name: PC1, dtype: float32


In [16]:
import numpy as np

loadings_pc1 = pca_std.components_[0]  # weight of each SNP on PC1
top_n = 20
top_idx = np.argsort(np.abs(loadings_pc1))[::-1][:top_n]

# need probe IDs aligned to the valid_snp_mask filtering we did
probe_ids_after_autosomal = probe_id_array[keep_mask]
probe_ids_valid = probe_ids_after_autosomal[valid_snp_mask]

top_probes_pc1 = probe_ids_valid[top_idx]
top_loadings = loadings_pc1[top_idx]

for pid, load in zip(top_probes_pc1, top_loadings):
    core = strip_address_suffix(pid)
    match = manifest_df[manifest_df["IlmnID"].apply(strip_address_suffix) == core]
    chrom = match["Chr"].values[0] if len(match) else "?"
    print(f"{pid}  loading={load:.4f}  Chr={chrom}")

exm2259823-0_B_F_1975243199  loading=0.0279  Chr=1
exm2260965-0_B_F_1975247368  loading=0.0250  Chr=20
exm2266920-0_T_F_1984851404  loading=0.0246  Chr=9
exm2260814-0_T_F_1975256880  loading=0.0244  Chr=18
exm2264198-0_T_F_1984848214  loading=0.0240  Chr=6
exm2267597-0_T_F_1984846997  loading=0.0237  Chr=13
exm2262262-0_B_R_1975241150  loading=0.0235  Chr=7
exm2261750-0_B_F_1975265406  loading=0.0233  Chr=4
exm2260437-0_T_F_1975263602  loading=0.0233  Chr=15
exm2262606-0_T_F_1975246109  loading=0.0228  Chr=9
exm2267714-0_T_F_1984853815  loading=0.0228  Chr=14
exm2261086-0_B_F_2058868776  loading=0.0228  Chr=2
exm2262500-0_T_R_1975264765  loading=0.0228  Chr=8
exm2267831-0_B_R_1984854044  loading=0.0227  Chr=15
exm2259686-0_T_F_1975240899  loading=0.0227  Chr=11
exm2265824-0_B_R_1984855918  loading=0.0226  Chr=4
exm732028-0_B_R_1922961380  loading=0.0226  Chr=8
exm2262385-0_B_R_1975252095  loading=0.0225  Chr=7
exm2271000-0_B_F_1984854901  loading=0.0225  Chr=8
exm1159330-0_B_R_19228472

In [17]:
# sanity check: PCA was computed purely from genotype data, no phenotype involved
print("Was smoking_status used as PCA input? No — PCA fit only on X_standardized (genotype-derived).")
print("X_standardized shape used for PCA:", X_standardized.shape, "(samples x SNPs only)")

Was smoking_status used as PCA input? No — PCA fit only on X_standardized (genotype-derived).
X_standardized shape used for PCA: (3348, 141645) (samples x SNPs only)


In [18]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from scipy import stats

# align smoking_status (binary 0/1) to sample_ids order
meta_df_check2 = pd.read_csv(r"C:\Users\user\Downloads\GSE148375_clean\checkpoint2_metadata_sample_filtered.csv")
meta_df_check2["sample_id"] = meta_df_check2["sample_id"].astype(str)
status_map = meta_df_check2.set_index("sample_id")["smoking_status"]

Y = np.array([1 if status_map.get(sid) == "Smoker" else 0 for sid in sample_ids])
print("Y distribution:", np.unique(Y, return_counts=True))

X_confounders = pcs_std  # our 10 PCs, samples x 10

def doubleml_single_snp(D, Y, X_confounders):
    """
    D: SNP genotype vector (n,)
    Y: outcome vector (n,)
    X_confounders: PCs matrix (n, 10)
    Returns: theta (causal effect estimate), p-value
    """
    # Model 1: predict D from confounders
    model_D = LinearRegression()
    model_D.fit(X_confounders, D)
    D_resid = D - model_D.predict(X_confounders)

    # Model 2: predict Y from confounders
    model_Y = LinearRegression()
    model_Y.fit(X_confounders, Y)
    Y_resid = Y - model_Y.predict(X_confounders)

    # Final step: regress Y_resid on D_resid
    n = len(D_resid)
    D_resid_with_const = np.column_stack([np.ones(n), D_resid])
    theta_hat = np.linalg.lstsq(D_resid_with_const, Y_resid, rcond=None)[0][1]

    # standard error and p-value (simple OLS-based)
    residuals = Y_resid - D_resid_with_const @ np.linalg.lstsq(D_resid_with_const, Y_resid, rcond=None)[0]
    sigma2 = np.sum(residuals**2) / (n - 2)
    var_theta = sigma2 / np.sum((D_resid - D_resid.mean())**2)
    se_theta = np.sqrt(var_theta)
    t_stat = theta_hat / se_theta
    p_value = 2 * (1 - stats.t.cdf(np.abs(t_stat), df=n-2))

    return theta_hat, p_value

# test on one SNP
test_snp_id = probe_ids_valid[0]
test_snp_idx = 0
D_test = X_standardized[:, test_snp_idx]

theta, pval = doubleml_single_snp(D_test, Y, X_confounders)
print(f"Test SNP: {test_snp_id}")
print(f"Theta (causal effect estimate): {theta:.6f}")
print(f"P-value: {pval:.6f}")

Y distribution: (array([0, 1]), array([1717, 1631]))
Test SNP: exm2268640-0_B_F_1984844585
Theta (causal effect estimate): 0.000700
P-value: 0.634268


In [19]:
from sklearn.model_selection import KFold

def doubleml_single_snp_crossfit(D, Y, X_confounders, n_folds=5, random_state=42):
    n = len(D)
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=random_state)

    D_resid = np.zeros(n)
    Y_resid = np.zeros(n)

    for train_idx, test_idx in kf.split(X_confounders):
        model_D = LinearRegression().fit(X_confounders[train_idx], D[train_idx])
        D_resid[test_idx] = D[test_idx] - model_D.predict(X_confounders[test_idx])

        model_Y = LinearRegression().fit(X_confounders[train_idx], Y[train_idx])
        Y_resid[test_idx] = Y[test_idx] - model_Y.predict(X_confounders[test_idx])

    D_resid_with_const = np.column_stack([np.ones(n), D_resid])
    coefs = np.linalg.lstsq(D_resid_with_const, Y_resid, rcond=None)[0]
    theta_hat = coefs[1]

    residuals = Y_resid - D_resid_with_const @ coefs
    sigma2 = np.sum(residuals**2) / (n - 2)
    var_theta = sigma2 / np.sum((D_resid - D_resid.mean())**2)
    se_theta = np.sqrt(var_theta)
    t_stat = theta_hat / se_theta
    p_value = 2 * (1 - stats.t.cdf(np.abs(t_stat), df=n-2))

    return theta_hat, p_value

theta_cf, pval_cf = doubleml_single_snp_crossfit(D_test, Y, X_confounders)
print(f"Cross-fitted theta: {theta_cf:.6f}")
print(f"Cross-fitted p-value: {pval_cf:.6f}")

Cross-fitted theta: 0.000601
Cross-fitted p-value: 0.685968


In [20]:
import numpy as np
from sklearn.model_selection import KFold

def doubleml_full_scan(X_snps, Y, X_confounders, n_folds=5, random_state=42):
    """
    X_snps: (n_samples, n_snps) standardized genotype matrix
    Y: (n_samples,) outcome
    X_confounders: (n_samples, n_pcs)
    Returns: theta array, pvalue array (one per SNP)
    """
    n, n_snps = X_snps.shape
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=random_state)

    D_resid_all = np.zeros_like(X_snps)
    Y_resid = np.zeros(n)

    for train_idx, test_idx in kf.split(X_confounders):
        Xc_train, Xc_test = X_confounders[train_idx], X_confounders[test_idx]

        # add intercept column
        Xc_train_i = np.column_stack([np.ones(len(train_idx)), Xc_train])
        Xc_test_i = np.column_stack([np.ones(len(test_idx)), Xc_test])

        # --- residualize Y (single regression) ---
        coef_Y = np.linalg.lstsq(Xc_train_i, Y[train_idx], rcond=None)[0]
        Y_resid[test_idx] = Y[test_idx] - Xc_test_i @ coef_Y

        # --- residualize ALL SNPs at once (vectorized least squares) ---
        # solves for coefficients of all SNP columns simultaneously
        coef_D_all = np.linalg.lstsq(Xc_train_i, X_snps[train_idx], rcond=None)[0]  # (n_pcs+1, n_snps)
        D_resid_all[test_idx] = X_snps[test_idx] - Xc_test_i @ coef_D_all

    # now compute theta and p-value per SNP, vectorized
    Y_resid_centered = Y_resid - Y_resid.mean()
    D_resid_centered = D_resid_all - D_resid_all.mean(axis=0)

    numerator = (D_resid_centered * Y_resid_centered[:, None]).sum(axis=0)
    denominator = (D_resid_centered ** 2).sum(axis=0)
    theta_hat_all = numerator / denominator

    # residual sum of squares per SNP for SE calculation
    fitted = D_resid_centered * theta_hat_all[None, :]
    resid_final = Y_resid_centered[:, None] - fitted
    sigma2_all = (resid_final ** 2).sum(axis=0) / (n - 2)
    var_theta_all = sigma2_all / denominator
    se_theta_all = np.sqrt(var_theta_all)

    t_stat_all = theta_hat_all / se_theta_all
    p_value_all = 2 * (1 - stats.t.cdf(np.abs(t_stat_all), df=n-2))

    return theta_hat_all, p_value_all

# quick test on a small subset first (e.g. 1000 SNPs) before running on all 233,610
test_subset = X_standardized[:, :1000]
theta_subset, pval_subset = doubleml_full_scan(test_subset, Y.astype(float), X_confounders)

print("Theta range:", theta_subset.min(), theta_subset.max())
print("P-value summary:", pd.Series(pval_subset).describe())

# cross-check: does SNP 0's result match our earlier single-SNP cross-fitted computation?
print("SNP 0 theta (vectorized):", theta_subset[0], " vs single-SNP:", theta_cf)
print("SNP 0 pval (vectorized):", pval_subset[0], " vs single-SNP:", pval_cf)

Theta range: -0.2869767876333973 0.18269288375818138
P-value summary: count    1.000000e+03
mean     4.094060e-01
std      2.907786e-01
min      2.807887e-08
25%      1.620051e-01
50%      3.647478e-01
75%      6.347639e-01
max      9.996216e-01
dtype: float64
SNP 0 theta (vectorized): 0.0006011415984528173  vs single-SNP: 0.0006011416049912311
SNP 0 pval (vectorized): 0.685968436831579  vs single-SNP: 0.6859684346700867


In [2]:
import pandas as pd
import numpy as np

out_dir = r"C:\Users\user\Downloads\GSE148375_clean"

# reload encoded SNP matrix (SNPs x samples)
encoded_df = pd.read_csv(out_dir + r"\checkpoint7_snp_encoded_012.csv")
probe_id_array = encoded_df["probe_id"].to_numpy()
sample_cols = encoded_df.columns[1:]
sample_ids = sample_cols.tolist()

X_snp_first = encoded_df[sample_cols].to_numpy(dtype=np.int8)
print("X_snp_first shape:", X_snp_first.shape)

# reload manifest, rebuild sex-linked exclusion list
manifest_path = r"C:\Users\user\Downloads\HumanExome-12-v1-0-B.csv"
manifest_df = pd.read_csv(manifest_path, skiprows=7, low_memory=False)

import re
def strip_address_suffix(probe_id):
    return re.sub(r'_\d+$', '', probe_id)

non_autosomal = {"X", "Y", "XY", "MT", "0"}
sex_linked_probes = set(manifest_df.loc[manifest_df["Chr"].isin(non_autosomal), "IlmnID"])
sex_linked_core_names = {strip_address_suffix(p) for p in sex_linked_probes}

our_probe_core_names = {strip_address_suffix(p): p for p in probe_id_array}
to_exclude = set()
for core in sex_linked_core_names:
    if core in our_probe_core_names:
        to_exclude.add(our_probe_core_names[core])

keep_mask = ~np.isin(probe_id_array, list(to_exclude))
print("Probes to keep (autosomal):", keep_mask.sum())

X_snp_first_auto = X_snp_first[keep_mask]
X_auto = X_snp_first_auto.T  # samples x SNPs
print("X_auto shape:", X_auto.shape)

X_snp_first shape: (238927, 3348)
Probes to keep (autosomal): 233610
X_auto shape: (3348, 233610)


In [3]:
import numpy as np
from sklearn.decomposition import PCA
import gc

X_auto_float = X_auto.astype(np.float32)
del X_snp_first, X_snp_first_auto, X_auto
gc.collect()

p = X_auto_float.mean(axis=0) / 2
denom = np.sqrt(2 * p * (1 - p))
valid_snp_mask = denom > 1e-8
print("Monomorphic SNPs to exclude:", (~valid_snp_mask).sum())

X_valid = X_auto_float[:, valid_snp_mask]
p_valid = p[valid_snp_mask]
denom_valid = denom[valid_snp_mask]

X_standardized = (X_valid - 2 * p_valid) / denom_valid
del X_auto_float, X_valid
gc.collect()

print("X_standardized shape:", X_standardized.shape)

pca_std = PCA(n_components=10, random_state=42, copy=False, svd_solver='randomized')
pcs_std = pca_std.fit_transform(X_standardized)

print("PC scores shape:", pcs_std.shape)
print("Explained variance ratio:", pca_std.explained_variance_ratio_)

Monomorphic SNPs to exclude: 91965
X_standardized shape: (3348, 141645)
PC scores shape: (3348, 10)
Explained variance ratio: [0.00563623 0.00111008 0.00105167 0.00098916 0.00098035 0.00095548
 0.00091716 0.00086775 0.00085743 0.00084649]


In [4]:
import pandas as pd

meta_df_check2 = pd.read_csv(r"C:\Users\user\Downloads\GSE148375_clean\checkpoint2_metadata_sample_filtered.csv")
meta_df_check2["sample_id"] = meta_df_check2["sample_id"].astype(str)
status_map = meta_df_check2.set_index("sample_id")["smoking_status"]

Y = np.array([1 if status_map.get(sid) == "Smoker" else 0 for sid in sample_ids])
print("Y distribution:", np.unique(Y, return_counts=True))

X_confounders = pcs_std

probe_ids_after_autosomal = probe_id_array[keep_mask]
probe_ids_valid = probe_ids_after_autosomal[valid_snp_mask]
print("probe_ids_valid count:", len(probe_ids_valid))

Y distribution: (array([0, 1]), array([1717, 1631]))
probe_ids_valid count: 141645


In [5]:
import numpy as np
from sklearn.model_selection import KFold
from scipy import stats
import time

def doubleml_full_scan_v2(X_snps, Y, X_confounders, n_folds=5, random_state=42):
    n, n_snps = X_snps.shape
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=random_state)

    D_resid_all = np.zeros_like(X_snps)
    Y_resid = np.zeros(n)

    for train_idx, test_idx in kf.split(X_confounders):
        Xc_train, Xc_test = X_confounders[train_idx], X_confounders[test_idx]
        Xc_train_i = np.column_stack([np.ones(len(train_idx)), Xc_train])
        Xc_test_i = np.column_stack([np.ones(len(test_idx)), Xc_test])

        coef_Y = np.linalg.lstsq(Xc_train_i, Y[train_idx], rcond=None)[0]
        Y_resid[test_idx] = Y[test_idx] - Xc_test_i @ coef_Y

        coef_D_all = np.linalg.lstsq(Xc_train_i, X_snps[train_idx], rcond=None)[0]
        D_resid_all[test_idx] = X_snps[test_idx] - Xc_test_i @ coef_D_all

    Y_resid_centered = Y_resid - Y_resid.mean()
    D_resid_centered = D_resid_all - D_resid_all.mean(axis=0)
    del D_resid_all

    sum_DY = (D_resid_centered * Y_resid_centered[:, None]).sum(axis=0)
    sum_DD = (D_resid_centered ** 2).sum(axis=0)
    sum_YY = (Y_resid_centered ** 2).sum()
    del D_resid_centered

    theta_hat_all = sum_DY / sum_DD
    ssr_all = sum_YY - (sum_DY ** 2) / sum_DD
    sigma2_all = ssr_all / (n - 2)
    var_theta_all = sigma2_all / sum_DD
    se_theta_all = np.sqrt(var_theta_all)

    t_stat_all = theta_hat_all / se_theta_all
    p_value_all = 2 * (1 - stats.t.cdf(np.abs(t_stat_all), df=n-2))

    return theta_hat_all, p_value_all

start = time.time()
theta_all, pval_all = doubleml_full_scan_v2(X_standardized, Y.astype(float), X_confounders)
elapsed = time.time() - start

print(f"Completed in {elapsed:.1f} seconds")
print("Theta range:", theta_all.min(), theta_all.max())
print(pd.Series(pval_all).describe())
print("SNPs with raw p < 0.05:", (pval_all < 0.05).sum())
print("SNPs with raw p < 0.001:", (pval_all < 0.001).sum())

Completed in 61.6 seconds
Theta range: -0.11128879806544378 0.2478147604240732
count    141645.000000
mean          0.462100
std           0.285376
min           0.000006
25%           0.242436
50%           0.393523
75%           0.693430
max           0.999976
dtype: float64
SNPs with raw p < 0.05: 6598
SNPs with raw p < 0.001: 128


In [6]:
! pip install statsmodels

In [7]:
from statsmodels.stats.multitest import multipletests

reject, pvals_corrected, _, _ = multipletests(pval_all, alpha=0.05, method='fdr_bh')

print("Number of SNPs significant after BH-FDR correction (alpha=0.05):", reject.sum())
print("Min corrected p-value:", pvals_corrected.min())
print("Number with corrected p < 0.1:", (pvals_corrected < 0.1).sum())
print("Number with corrected p < 0.01:", (pvals_corrected < 0.01).sum())

Number of SNPs significant after BH-FDR correction (alpha=0.05): 0
Min corrected p-value: 0.3541843401374489
Number with corrected p < 0.1: 0
Number with corrected p < 0.01: 0


In [8]:
import numpy as np
import time

n_repeats = 30
threshold = 0.001
n_snps = X_standardized.shape[1]

significant_counts = np.zeros(n_snps, dtype=int)

start = time.time()
for rep in range(n_repeats):
    theta_rep, pval_rep = doubleml_full_scan_v2(X_standardized, Y.astype(float), X_confounders, random_state=rep)
    significant_counts += (pval_rep < threshold).astype(int)
    if (rep + 1) % 5 == 0:
        print(f"Completed {rep+1}/{n_repeats} repeats, elapsed {time.time()-start:.1f}s")

print(f"Total time: {time.time()-start:.1f}s")

stability_fraction = significant_counts / n_repeats
print("Distribution of stability fraction:")
print(pd.Series(stability_fraction).describe())
print("SNPs significant in >=80% of repeats:", (stability_fraction >= 0.8).sum())
print("SNPs significant in >=50% of repeats:", (stability_fraction >= 0.5).sum())

Completed 5/30 repeats, elapsed 381.9s
Completed 10/30 repeats, elapsed 753.7s
Completed 15/30 repeats, elapsed 1126.1s
Completed 20/30 repeats, elapsed 1495.4s
Completed 25/30 repeats, elapsed 1865.5s
Completed 30/30 repeats, elapsed 2236.5s
Total time: 2236.5s
Distribution of stability fraction:
count    141645.000000
mean          0.000885
std           0.027874
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max           1.000000
dtype: float64
SNPs significant in >=80% of repeats: 105
SNPs significant in >=50% of repeats: 119


In [9]:
import os

out_dir = r"C:\Users\user\Downloads\GSE148375_clean"

stability_df = pd.DataFrame({
    "probe_id": probe_ids_valid,
    "stability_fraction": stability_fraction,
    "n_significant_repeats": significant_counts
})
stability_df = stability_df.sort_values("stability_fraction", ascending=False)

stability_df.to_csv(os.path.join(out_dir, "checkpoint9_doubleml_stability_results.csv"), index=False)
print("Saved stability selection results.")

shortlist_80 = stability_df[stability_df["stability_fraction"] >= 0.8].copy()
print("Shortlist size (>=80% stability):", len(shortlist_80))
print(shortlist_80.head(15))

Saved stability selection results.
Shortlist size (>=80% stability): 105
                                 probe_id  stability_fraction  \
91890         exm1000949-0_B_F_1922459089                 1.0   
139613        exm1597603-0_T_R_2060425485                 1.0   
10975          exm131347-0_T_R_2060139713                 1.0   
48344          exm528294-0_B_F_1921884303                 1.0   
49676    exm-rs9501610-131_B_F_2058871306                 1.0   
33552          exm381774-0_T_R_1923133868                 1.0   
137025        exm1563189-0_T_R_1918426074                 1.0   
136997   exm-rs2823962-131_T_F_1990490949                 1.0   
12209    exm-rs4844614-131_B_F_1990483625                 1.0   
136279  exm-rs16982520-131_T_F_1990489099                 1.0   
57236          exm609218-0_B_F_1918575531                 1.0   
135746        exm1547532-0_B_F_1922662889                 1.0   
6463          exm2249471-0_B_F_1975256234                 1.0   
24626          ex

In [10]:
import pandas as pd

manifest_path = r"C:\Users\user\Downloads\HumanExome-12-v1-0-B.csv"
manifest_df = pd.read_csv(manifest_path, skiprows=7, low_memory=False)

import re
def strip_address_suffix(probe_id):
    return re.sub(r'_\d+$', '', probe_id)

manifest_df["core_name"] = manifest_df["IlmnID"].map(strip_address_suffix)

shortlist_80 = shortlist_80.copy()
shortlist_80["core_name"] = shortlist_80["probe_id"].map(strip_address_suffix)

position_lookup = manifest_df.set_index("core_name")[["Chr", "MapInfo"]]
shortlist_with_pos = shortlist_80.merge(position_lookup, left_on="core_name", right_index=True, how="left")

print(shortlist_with_pos[["probe_id", "Chr", "MapInfo", "stability_fraction"]].sort_values(["Chr", "MapInfo"]))
print("Unresolved positions:", shortlist_with_pos["Chr"].isna().sum())

                          probe_id Chr      MapInfo  stability_fraction
1555     exm21541-0_T_R_1921517700   1   16531263.0            0.866667
4691     exm61019-0_T_R_1919145850   1   54476084.0            0.933333
6463   exm2249471-0_B_F_1975256234   1  102389733.0            1.000000
8199    exm100944-0_B_R_1921482882   1  152329460.0            1.000000
10967   exm131290-0_B_R_1921524890   1  185956648.0            1.000000
...                            ...  ..          ...                 ...
67523   exm724762-0_B_R_1922905822   8  142506444.0            1.000000
70124  exm2271094-0_T_R_1984851691   9   80515456.0            1.000000
70366   exm759751-0_T_R_1922412944   9   90584110.0            1.000000
72060   exm777999-0_B_F_1922426645   9  123914765.0            1.000000
72459   exm782555-0_B_R_1922302526   9  130206472.0            1.000000

[105 rows x 4 columns]
Unresolved positions: 0


In [11]:
import pandas as pd
import numpy as np

shortlist_sorted = shortlist_with_pos.sort_values(["Chr", "MapInfo"]).reset_index(drop=True)

ld_window_bp = 1_000_000  # 1 Mb window, standard upper bound for LD checks

close_pairs = []
for chrom in shortlist_sorted["Chr"].unique():
    sub = shortlist_sorted[shortlist_sorted["Chr"] == chrom].sort_values("MapInfo")
    positions = sub["MapInfo"].values
    probe_ids_chrom = sub["probe_id"].values
    for i in range(len(positions)):
        for j in range(i+1, len(positions)):
            dist = positions[j] - positions[i]
            if dist > ld_window_bp:
                break  # sorted, so no need to check further
            close_pairs.append((probe_ids_chrom[i], probe_ids_chrom[j], chrom, dist))

print("Number of SNP pairs within 1Mb of each other:", len(close_pairs))
for pair in close_pairs:
    print(pair)

Number of SNP pairs within 1Mb of each other: 39
('exm131290-0_B_R_1921524890', 'exm131347-0_T_R_2060139713', '1', np.float64(16292.0))
('exm131290-0_B_R_1921524890', 'exm131535-0_B_R_1921434966', '1', np.float64(93769.0))
('exm131347-0_T_R_2060139713', 'exm131535-0_B_R_1921434966', '1', np.float64(77477.0))
('exm143836-0_T_F_1921339047', 'exm-rs4844614-131_B_F_1990483625', '1', np.float64(902868.0))
('exm923301-0_T_R_1918207126', 'exm2250129-0_B_F_1975250412', '11', np.float64(0.0))
('exm1244011-0_T_F_1921784787', 'exm1245580-0_B_F_2060131617', '16', np.float64(722918.0))
('exm1350304-0_B_R_1918618514', 'exm1350497-0_T_R_1918721584', '17', np.float64(35878.0))
('exm1350304-0_B_R_1918618514', 'exm1350506-0_T_R_1918789000', '17', np.float64(35998.0))
('exm1350304-0_B_R_1918618514', 'exm1350514-0_B_F_1918693303', '17', np.float64(36072.0))
('exm1350497-0_T_R_1918721584', 'exm1350506-0_T_R_1918789000', '17', np.float64(120.0))
('exm1350497-0_T_R_1918721584', 'exm1350514-0_B_F_1918693303',

In [12]:
import numpy as np

# get encoded genotype values (0/1/2) for shortlisted SNPs from X_standardized
# need to map probe_id -> column index in X_standardized (which uses probe_ids_valid)
probe_id_to_idx = {pid: i for i, pid in enumerate(probe_ids_valid)}

def get_genotype_vector(probe_id):
    idx = probe_id_to_idx[probe_id]
    return X_standardized[:, idx]

# compute r² for all close pairs
pair_results = []
for pid1, pid2, chrom, dist in close_pairs:
    if pid1 not in probe_id_to_idx or pid2 not in probe_id_to_idx:
        continue
    g1 = get_genotype_vector(pid1)
    g2 = get_genotype_vector(pid2)
    r = np.corrcoef(g1, g2)[0, 1]
    r2 = r ** 2
    pair_results.append((pid1, pid2, chrom, dist, r2))

pair_df = pd.DataFrame(pair_results, columns=["probe1", "probe2", "chr", "dist_bp", "r2"])
pair_df = pair_df.sort_values("r2", ascending=False)
print(pair_df.to_string())
print("\nPairs with r² > 0.2:", (pair_df["r2"] > 0.2).sum())
print("Pairs with r² > 0.5:", (pair_df["r2"] > 0.5).sum())
print("Pairs with r² > 0.8:", (pair_df["r2"] > 0.8).sum())

                              probe1                            probe2 chr   dist_bp        r2
38  exm-rs9262135-131_B_R_1990485844        exm528294-0_B_F_1921884303   6   33875.0  1.000000
11       exm1350506-0_T_R_1918789000       exm1350514-0_B_F_1918693303  17      74.0  0.998795
4         exm923301-0_T_R_1918207126       exm2250129-0_B_F_1975250412  11       0.0  0.994623
9        exm1350497-0_T_R_1918721584       exm1350506-0_T_R_1918789000  17     120.0  0.992831
10       exm1350497-0_T_R_1918721584       exm1350514-0_B_F_1918693303  17     194.0  0.991634
29  exm-rs2239529-131_B_F_1990480963  exm-rs2523987-131_T_R_1990490600   6    1663.0  0.931408
6        exm1350304-0_B_R_1918618514       exm1350497-0_T_R_1918721584  17   35878.0  0.918206
7        exm1350304-0_B_R_1918618514       exm1350506-0_T_R_1918789000  17   35998.0  0.911346
8        exm1350304-0_B_R_1918618514       exm1350514-0_B_F_1918693303  17   36072.0  0.910193
0         exm131290-0_B_R_1921524890        exm131

In [13]:
import numpy as np
import pandas as pd
import os

r2_threshold = 0.2
window_bp = 1_000_000

# Step 1: sort by stability fraction descending (keep the most stable first)
shortlist_sorted_stab = shortlist_with_pos.sort_values(
    "stability_fraction", ascending=False
).reset_index(drop=True)

retained = []
removed = set()

for i, row_i in shortlist_sorted_stab.iterrows():
    pid_i = row_i["probe_id"]
    if pid_i in removed:
        continue

    # keep this SNP
    retained.append(pid_i)

    # remove every subsequent SNP within 1Mb AND r2 > threshold with this one
    chr_i = row_i["Chr"]
    pos_i = row_i["MapInfo"]
    g_i = get_genotype_vector(pid_i)

    for j, row_j in shortlist_sorted_stab.iloc[i+1:].iterrows():
        pid_j = row_j["probe_id"]
        if pid_j in removed:
            continue
        if row_j["Chr"] != chr_i:
            continue
        if abs(row_j["MapInfo"] - pos_i) > window_bp:
            continue

        # within window — check actual r²
        g_j = get_genotype_vector(pid_j)
        r2 = np.corrcoef(g_i, g_j)[0, 1] ** 2
        if r2 > r2_threshold:
            removed.add(pid_j)

print("SNPs retained after greedy LD pruning:", len(retained))
print("SNPs removed:", len(removed))

shortlist_pruned = shortlist_with_pos[
    shortlist_with_pos["probe_id"].isin(retained)
].copy()
shortlist_pruned = shortlist_pruned.sort_values(["Chr", "MapInfo"]).reset_index(drop=True)

print(shortlist_pruned[["probe_id", "Chr", "MapInfo", "stability_fraction"]])

out_dir = r"C:\Users\user\Downloads\GSE148375_clean"
shortlist_pruned.to_csv(os.path.join(out_dir, "checkpoint10_shortlist_ld_pruned.csv"), index=False)
print("Saved LD-pruned shortlist.")

SNPs retained after greedy LD pruning: 96
SNPs removed: 9
                       probe_id Chr      MapInfo  stability_fraction
0     exm21541-0_T_R_1921517700   1   16531263.0            0.866667
1     exm61019-0_T_R_1919145850   1   54476084.0            0.933333
2   exm2249471-0_B_F_1975256234   1  102389733.0            1.000000
3    exm100944-0_B_R_1921482882   1  152329460.0            1.000000
4    exm131347-0_T_R_2060139713   1  185972940.0            1.000000
..                          ...  ..          ...                 ...
91   exm724762-0_B_R_1922905822   8  142506444.0            1.000000
92  exm2271094-0_T_R_1984851691   9   80515456.0            1.000000
93   exm759751-0_T_R_1922412944   9   90584110.0            1.000000
94   exm777999-0_B_F_1922426645   9  123914765.0            1.000000
95   exm782555-0_B_R_1922302526   9  130206472.0            1.000000

[96 rows x 4 columns]
Saved LD-pruned shortlist.
